# Implementaciones independientes por plataforma: QDSV Bridge vs Qrisp nativo

Este notebook v19 compara implementaciones específicas de cada plataforma construidas exclusivamente mediante sus respectivas interfaces públicas:

- **QDSV:** una capa local reutilizable forma la especificación pública; la generación del circuito se delega exclusivamente a `qdsv-bridge`.
- **Qrisp:** infraestructura local prepara los datos reversibles y cada regla empresarial se expresa manualmente mediante `QuantumFloat`, `QuantumBool` y `ConditionEnvironment`.
- No se intercambian QASM, QPY, registros, ancillas ni artefactos entre plataformas.

Ambas plataformas comparten necesariamente los datos y el significado semántico público del problema. El ground truth permanece fuera de sus builders y se utiliza únicamente en el harness posterior de replay y métricas.

El esfuerzo se atribuye en categorías separadas: definición neutral del problema, infraestructura reutilizable, lógica específica por caso e invocación visible. No se afirma que ninguna plataforma opere "sin adaptador" o sin código de integración.

Track Qrisp: **Qrisp 0.9.6 native public API + audited packaging compatibility shim**.


## 1. Instalación reproducible en Colab


In [ ]:
#@title Instalar entorno congelado
import subprocess, sys

SUPPORTED_PYTHON = {(3, 11), (3, 12)}
if sys.version_info[:2] not in SUPPORTED_PYTHON:
    raise RuntimeError(
        f"Este benchmark exige Python 3.11 o 3.12; runtime actual: "
        f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}"
    )

PACKAGES = [
    "qdsv-bridge[qiskit]==0.6.1",
    "qrisp==0.9.6",
    "qiskit==2.5.1",
    "qiskit-aer==0.17.2",
    "qiskit-qasm3-import>=0.5,<0.7",
    "numpy==2.0.2",
    "pandas==2.2.2",
    "sympy==1.13.0",
    "packaging==23.2",
    "tabulate==0.9.0",
    "psutil==6.1.1",
]
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--no-cache-dir", "-q", *PACKAGES
])
print("Instalación terminada.")
print("Si el runtime ya había importado alguna dependencia, reinícialo y continúa desde la siguiente celda.")


In [ ]:
#@title Verificar y congelar el entorno efectivo
import importlib.metadata as md
import json, platform, subprocess, sys
from datetime import datetime, timezone

EXPECTED = {
    "qdsv-bridge": "0.6.1",
    "qrisp": "0.9.6",
    "qiskit": "2.5.1",
    "qiskit-aer": "0.17.2",
    "numpy": "2.0.2",
    "pandas": "2.2.2",
    "sympy": "1.13.0",
    "packaging": "23.2",
    "psutil": "6.1.1",
}
observed_versions = {}
for package, expected in EXPECTED.items():
    installed = md.version(package)
    observed_versions[package] = installed
    print(f"{package}: {installed}")
    assert installed == expected, f"Se esperaba {package}=={expected}; instalado: {installed}"

assert sys.version_info[:2] in {(3, 11), (3, 12)}
print("python:", platform.python_version())

completed = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    text=True,
    capture_output=True,
)
print("\n--- pip check ---")
pip_check_text = completed.stdout or completed.stderr or "Sin conflictos declarados por pip check."
print(pip_check_text)

CORE_DISTRIBUTIONS = {
    "qdsv-bridge", "qrisp", "qiskit", "qiskit-aer",
    "qiskit-qasm3-import", "numpy", "pandas",
}
pip_check_lines = [
    line.strip() for line in pip_check_text.splitlines() if line.strip()
]
core_conflicts = [
    line for line in pip_check_lines
    if line.split()[0].lower() in CORE_DISTRIBUTIONS
]
external_conflicts = [
    line for line in pip_check_lines if line not in core_conflicts
]
if core_conflicts:
    raise RuntimeError(
        "pip check detectó incompatibilidades declaradas por paquetes del benchmark:\n"
        + "\n".join(core_conflicts)
    )
if completed.returncode != 0:
    print(
        "Advertencia: existen conflictos declarados por otros paquetes preinstalados "
        "de Colab. Se conservarán en environment.json, pero no se atribuyen al benchmark."
    )

PIP_FREEZE = subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze", "--all"], text=True
)
ENVIRONMENT_SNAPSHOT = {
    "captured_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "implementation": platform.python_implementation(),
    "platform": platform.platform(),
    "expected_versions": EXPECTED,
    "observed_versions": observed_versions,
    "pip_check_returncode": completed.returncode,
    "pip_check_lines": pip_check_lines,
    "core_conflicts": core_conflicts,
    "external_conflicts": external_conflicts,
}
print("Entorno congelado en memoria; se guardará dentro del run_id.")


## 2. Configuración y controles de ejecución

In [ ]:
import os, io, re, json, math, time, random, hashlib, traceback, textwrap, shutil
import multiprocessing as mp
import queue as queue_module
import platform, pprint, uuid, psutil
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from qiskit import qasm2, qasm3, transpile
from qiskit_aer import AerSimulator

SEED = 20260805
SHOTS = 8192
CANDIDATE_DISTRIBUTION_ALPHA = 0.01
ERROR_EVENT_FAMILY_ALPHA = 0.05
ERROR_EVENT_COUNT = 3  # mismatch, masa inválida y ancillas sucias
MAX_EXACT_QUBITS = 0  # Fuerza MPS común para ambas plataformas
COMMON_BASIS = ["rz", "sx", "x", "cx"]
COMMON_OPT_LEVEL = 3

RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_NONCE = hashlib.sha256(f"{RUN_STAMP}:{uuid.uuid4()}".encode()).hexdigest()[:8]
RUN_ID = f"{RUN_STAMP}_{RUN_NONCE}"
OUTPUT_ROOT = Path("benchmark_qdsv_qrisp_v19_independent_native_runs")
OUTPUT_DIR = OUTPUT_ROOT / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
assert not any(OUTPUT_DIR.iterdir()), "El directorio de la corrida debe comenzar vacío."

TIMEOUT_SECONDS = {
    "README_SMOKE_CONSTRUCTION": 180,
    "QDSV_CONSTRUCTION": 300,
    "Qrisp_CONSTRUCTION": 1200,
    "REPLAY": 900,
    "NORMALIZATION": 900,
}
TIMEOUT_POLICY = {
    "version": "benchmark_timeout_policy.v2",
    "qrisp_construction_seconds": 1200,
    "historical_qrisp_reference_seconds": 900,
    "cross_track_timeout_comparison_allowed": False,
    "reason": "independent_public_native_implementations",
    "interpretation": "independent_builder_guardrail_only_not_compiler_speed",
}
MEMORY_LIMIT_MB = {
    "CONSTRUCTION": 6144,
    "REPLAY": 6144,
    "NORMALIZATION": 6144,
}
PROCESS_MONITOR_INTERVAL = 0.20

random.seed(SEED)
np.random.seed(SEED)

RUN_README_SMOKE = False  # Evita gastar una compilación Bridge adicional
RUN_QDSV = True
RUN_QRISP = True

# QDSV conserva el preflight de capacidades del servicio.
# Qrisp se ejecuta microcaso por microcaso con builders nativos explícitos.
# Un timeout conserva checkpoints y no bloquea los demás casos.
PLATFORM_PREFLIGHT_POLICY = {}

VERIFY_FROZEN_EXPECTATIONS = True


# Solo se aceptan hints explícitos y auditables.
REGISTER_HINTS: dict[tuple[str, str], dict[str, Any]] = {}

(OUTPUT_DIR / "environment.json").write_text(
    json.dumps(ENVIRONMENT_SNAPSHOT, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(OUTPUT_DIR / "requirements-freeze.txt").write_text(PIP_FREEZE, encoding="utf-8")

print("Semilla:", SEED)
print("Run ID:", RUN_ID)
print("Directorio de salida:", OUTPUT_DIR.resolve())
print("Timeouts:", TIMEOUT_SECONDS)
print("Memoria por etapa (MiB):", MEMORY_LIMIT_MB)


## 3. Contrato del benchmark y taxonomía de estados


In [ ]:
STATUS_DESCRIPTIONS = {
    "passed": "Artefacto materializado y equivalencia exacta aprobada.",
    "sampled_consistent": "Artefacto consistente con el contrato bajo el replay MPS muestreado prerregistrado.",
    "sampled_mismatch": "El replay MPS observó mismatch, masa inválida, ancillas sucias o preparación incompatible.",
    "semantic_mismatch": "El artefacto existe, pero no implementa el contrato esperado.",
    "unsupported": "La plataforma declara que la capacidad no está soportada.",
    "valid_rejection": "Rechazo explícito coherente con el contrato declarado.",
    "resource_rejected": "La construcción excede un límite de recursos.",
    "invalid_spec": "La plataforma rechazó la especificación pública como inválida.",
    "adapter_not_implemented": "El notebook no posee todavía un adaptador para ese caso.",
    "adapter_error": "Fallo en validación, traducción o código generado por el adaptador.",
    "framework_installation_error": "El paquete instalado no puede cargar un modulo requerido antes de construir el artefacto.",
    "framework_construction_error": "El framework falló después de recibir una construcción válida.",
    "service_error": "Error operativo de red o servicio.",
    "quota_exhausted": "La cuota pública se agotó antes de construir el caso.",
    "authorization_denied": "La cuenta fue autenticada, pero no está autorizada para sintetizar.",
    "artifact_unavailable": "Hubo construcción, pero no se obtuvo QASM/Qiskit verificable.",
    "artifact_export_error": "El artefacto exportado no pudo extraerse o parsearse.",
    "artifact_interchange_failed": "La síntesis terminó, pero no se obtuvo un artefacto de intercambio reproducible.",
    "replay_mapping_required": "No fue posible identificar inequívocamente los registros de salida.",
    "replay_resource_limited": "El circuito excede el límite lógico fijado para replay statevector exacto.",
    "replay_error": "El evaluador independiente falló después de cargar el artefacto.",
    "construction_timeout": "La construcción/materialización excedió el presupuesto temporal.",
    "construction_memory_limit": "La construcción/materialización excedió el presupuesto de memoria.",
    "replay_timeout": "El parsing o replay ideal excedió el presupuesto temporal.",
    "replay_memory_limit": "El parsing o replay ideal excedió el presupuesto de memoria.",
    "normalization_timeout": "La transpilación externa excedió el presupuesto temporal.",
    "normalization_memory_limit": "La transpilación externa excedió el presupuesto de memoria.",
    "worker_crash": "El proceso aislado terminó sin devolver un resultado.",
    "platform_preflight_blocked": "No se ejecutó el caso porque un canario previo detectó un bloqueo operacional común.",
    "replay_not_available": "El artefacto fue materializado, pero no se entregó contenido inline para replay externo.",
    "verification_incomplete": "La materialización existe, pero no pasaron todas las verificaciones externas requeridas.",
    "optimization_contract_incomplete": "El perfil fue solicitado, pero su ejecución no produjo evidencia completa.",
    "optimization_contract_failed": "La respuesta no cumplió el contrato congelado de optimización lógica.",
    "incomplete": "Existe evidencia parcial, insuficiente para decidir corrección.",
}

class AdapterError(RuntimeError):
    pass

class AdapterNotImplemented(AdapterError):
    pass

for key, value in STATUS_DESCRIPTIONS.items():
    print(f"{key:32s} {value}")


## 4. Perfiles de acceso público

Se reportan dos pruebas distintas:

- **`readme_quickstart`**: onboarding literal. Evalúa si el ejemplo inicial de una plataforma produce un artefacto correcto. No participa en cobertura ni recursos del benchmark principal.
- **`public_docs_full`**: benchmark principal. Los adaptadores pueden usar README, referencia API, tutoriales y ejemplos oficiales públicos, pero nunca código privado, respuestas congeladas ni reglas internas.

Esto evita comparar el README de una plataforma contra programación experta no documentada de otra.


In [ ]:
MAIN_ACCESS_PROFILE = "qdsv_qrisp_v19_independent_native"
ONBOARDING_ACCESS_PROFILE = "readme_quickstart"
BENCHMARK_VERSION_POLICY = "frozen_v19_independent_public_native_bilateral"
PLATFORM_ACCESS_PROFILES = {
    "QDSV": "public_docs_full",
    "Qrisp": "public_docs_native_condition_environment+audited_packaging_compatibility_shim",
}
ENVIRONMENT_SNAPSHOT["benchmark_version_policy"] = BENCHMARK_VERSION_POLICY
ENVIRONMENT_SNAPSHOT["platform_access_profiles"] = PLATFORM_ACCESS_PROFILES
(OUTPUT_DIR / "environment.json").write_text(
    json.dumps(ENVIRONMENT_SNAPSHOT, ensure_ascii=False, indent=2), encoding="utf-8"
)

PUBLIC_DOCUMENTATION_EVIDENCE = {
    "QDSV": [
        "https://github.com/qdsvquantum-afk/qdsv-bridge",
        "https://qdsvquantum-afk.github.io/qdsv-bridge/",
        "https://github.com/qdsvquantum-afk/qdsv-bridge/blob/main/notebooks/02_predicate_oracle_marking.ipynb",
    ],
    "Qrisp": [
        "https://qrisp.eu/reference/Quantum%20Types/QuantumFloat.html",
        "https://qrisp.eu/reference/Quantum%20Types/QuantumBool.html",
        "https://qrisp.eu/reference/Quantum%20Environments/ConditionEnvironment.html",
    ],
}
(OUTPUT_DIR / "documentation_profile.json").write_text(
    json.dumps(
        {
            "main_access_profile": MAIN_ACCESS_PROFILE,
            "onboarding_access_profile": ONBOARDING_ACCESS_PROFILE,
            "benchmark_version_policy": BENCHMARK_VERSION_POLICY,
            "platform_access_profiles": PLATFORM_ACCESS_PROFILES,
            "public_documentation_evidence": PUBLIC_DOCUMENTATION_EVIDENCE,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)
print("Perfil principal:", MAIN_ACCESS_PROFILE)
print("Onboarding separado:", ONBOARDING_ACCESS_PROFILE)


## 5. Onboarding literal de QDSV Bridge y catálogo público

Esta prueba reproduce el quickstart de Bridge como evidencia de onboarding del producto. Se excluye de las comparaciones cruzadas de cobertura, LOC y recursos. Después de definir el replay estricto se ejecutará con timeout y equivalencia dinámica.


In [ ]:
from qdsv_bridge import (
    QDSVBridgeClient,
    build_predicate_spec,
    select_recommended_artifact,
)

LOGICAL_OPTIMIZATION_CONTRACT = {
    "mode": "auto",
    "profile": "qiskit_structural_exact_v1",
    "acceptance_policy": "pareto_no_regression_v1",
}

README_SMOKE_SPEC = {
    "state_space": {"kind": "finite_candidates", "candidate_count": 2, "candidate_id": "candidate"},
    "signals": ["eligibility_score"],
    "prepared_candidates": [
        {"eligibility_score": 0},
        {"eligibility_score": 1},
    ],
    "goal": {
        "kind": "marking",
        "threshold": 1,
        "criteria": [
            {"signal": "eligibility_score", "importance": 1, "priority": 1}
        ],
    },
    "target": {
        "format": "qasm3",
        "backend_family": "qiskit",
        "logical_optimization": dict(LOGICAL_OPTIMIZATION_CONTRACT),
    },
    "limits": {"max_qubits": 8, "max_depth": 160},
}
README_SMOKE_EXPECTATION = {
    "true_indices": [1],
    "candidate_count": 2,
}

capability_catalog = None
try:
    capability_catalog = QDSVBridgeClient().capabilities()
    (OUTPUT_DIR / "qdsv_capabilities.json").write_text(
        json.dumps(capability_catalog, ensure_ascii=False, indent=2, default=repr),
        encoding="utf-8",
    )
    print("Catálogo de capacidades recibido.")
except Exception as exc:
    print("No se pudo consultar capabilities():", type(exc).__name__, str(exc))
    (OUTPUT_DIR / "qdsv_capabilities_error.txt").write_text(
        traceback.format_exc(), encoding="utf-8"
    )

canonical_capability_signal = None
if isinstance(capability_catalog, dict):
    for capability_key in ("canonical_generation", "predicate_generation", "generation"):
        candidate_signal = capability_catalog.get(capability_key)
        if isinstance(candidate_signal, dict):
            canonical_capability_signal = candidate_signal
            break
    if canonical_capability_signal is None and str(
        capability_catalog.get("status", "")
    ).lower() in {"unavailable", "disabled", "unsupported"}:
        canonical_capability_signal = {
            "status": capability_catalog.get("status"),
            "source": "capability_catalog_root",
        }

explicitly_unavailable = bool(
    isinstance(canonical_capability_signal, dict)
    and str(canonical_capability_signal.get("status", "")).lower()
    in {"unavailable", "disabled", "unsupported"}
)
QDSV_CANONICAL_PREFLIGHT = {
    "version": "qdsv_canonical_service_preflight.v2",
    "ok": not explicitly_unavailable,
    "blocking": explicitly_unavailable,
    "status": (
        "platform_preflight_blocked_explicitly_unavailable"
        if explicitly_unavailable
        else "passed"
        if isinstance(capability_catalog, dict)
        else "advisory_capability_catalog_unavailable"
    ),
    "catalog_consulted": isinstance(capability_catalog, dict),
    "canonical_capability_signal": canonical_capability_signal,
    "errors": (
        ["canonical_generation_explicitly_unavailable"]
        if explicitly_unavailable
        else []
        if isinstance(capability_catalog, dict)
        else ["capability_catalog_unavailable_generate_will_be_attempted"]
    ),
}

QDSV_OPTIMIZATION_PREFLIGHT = {
    "version": "qdsv_optional_optimization_preflight.v1",
    "ok": False,
    "status": "optional_capability_unavailable",
    "required_profile": LOGICAL_OPTIMIZATION_CONTRACT["profile"],
    "required_acceptance_policy": LOGICAL_OPTIMIZATION_CONTRACT["acceptance_policy"],
    "errors": [],
}
if isinstance(capability_catalog, dict):
    advertised = capability_catalog.get("logical_optimization") or {}
    if advertised.get("status") != "available":
        QDSV_OPTIMIZATION_PREFLIGHT["errors"].append("logical_optimization_not_available")
    if advertised.get("profile") != LOGICAL_OPTIMIZATION_CONTRACT["profile"]:
        QDSV_OPTIMIZATION_PREFLIGHT["errors"].append("optimization_profile_mismatch")
    if advertised.get("acceptance_policy") != LOGICAL_OPTIMIZATION_CONTRACT["acceptance_policy"]:
        QDSV_OPTIMIZATION_PREFLIGHT["errors"].append("acceptance_policy_mismatch")
else:
    QDSV_OPTIMIZATION_PREFLIGHT["errors"].append("capability_catalog_unavailable")
if not QDSV_OPTIMIZATION_PREFLIGHT["errors"]:
    QDSV_OPTIMIZATION_PREFLIGHT.update({"ok": True, "status": "passed"})

# Only an explicit canonical-unavailable signal blocks downstream generation.
QDSV_SERVICE_PREFLIGHT = QDSV_CANONICAL_PREFLIGHT
for filename, record in (
    ("qdsv_canonical_preflight.json", QDSV_CANONICAL_PREFLIGHT),
    ("qdsv_optimization_preflight.json", QDSV_OPTIMIZATION_PREFLIGHT),
):
    (OUTPUT_DIR / filename).write_text(
        json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8"
    )
print("Preflight canónico QDSV:", QDSV_CANONICAL_PREFLIGHT["status"])
print("Preflight opcional de optimización:", QDSV_OPTIMIZATION_PREFLIGHT["status"])
print("Smoke definido; permanece desactivado para no consumir otra compilación.")


## 6. Microcasos públicos congelados

Los cinco casos usan ocho candidatos y una sola comparación. Se modifica una dimensión a la vez: operador, anchura del campo o constante/campo en el lado derecho. No se incluyen labels ni respuestas en los builders.


In [ ]:
def F(name): return {"op": "field", "name": name}
def C(value): return {"op": "const", "value": value}
def CMP(op,a,b): return {"op": op, "left": a, "right": b}
def AND(*args): return {"op": "and", "args": list(args)}
def OR(*args): return {"op": "or", "args": list(args)}

BENCHMARK_CONTRACT_VERSION = "business_predicate.v3"

COMPLIANCE = [1, 1, 1, 0, 1, 1, 1, 1]
COST = [550, 590, 610, 500, 580, 450, 600, 599]
BUDGET = [600] * 8

PUBLIC_CASES = {
    "compliance_eq_constant_8": {
        "title": "Igualdad campo binario a constante",
        "business_family": "comparator_microdiagnostic",
        "candidate_count": 8,
        "data": {"compliance": COMPLIANCE},
        "predicate": CMP("eq", F("compliance"), C(1)),
        "qdsv_route": "bounded_predicate",
        "diagnostic_dimension": "equality_constant_binary_field",
    },
    "compliance_gte_constant_8": {
        "title": "Desigualdad campo binario a constante",
        "business_family": "comparator_microdiagnostic",
        "candidate_count": 8,
        "data": {"compliance": COMPLIANCE},
        "predicate": CMP("gte", F("compliance"), C(1)),
        "qdsv_route": "bounded_predicate",
        "diagnostic_dimension": "inequality_constant_binary_field",
    },
    "cost_eq_constant_8": {
        "title": "Igualdad campo ancho a constante",
        "business_family": "comparator_microdiagnostic",
        "candidate_count": 8,
        "data": {"cost": COST},
        "predicate": CMP("eq", F("cost"), C(550)),
        "qdsv_route": "bounded_predicate",
        "diagnostic_dimension": "equality_constant_wide_field",
    },
    "cost_lte_constant_8": {
        "title": "Desigualdad campo ancho a constante",
        "business_family": "comparator_microdiagnostic",
        "candidate_count": 8,
        "data": {"cost": COST},
        "predicate": CMP("lte", F("cost"), C(600)),
        "qdsv_route": "bounded_predicate",
        "diagnostic_dimension": "inequality_constant_wide_field",
    },
    "cost_lte_field_8": {
        "title": "Desigualdad campo ancho a campo",
        "business_family": "comparator_microdiagnostic",
        "candidate_count": 8,
        "data": {"cost": COST, "budget": BUDGET},
        "predicate": CMP("lte", F("cost"), F("budget")),
        "qdsv_route": "bounded_predicate",
        "diagnostic_dimension": "inequality_field_rhs",
    },
}

def ast_complexity(node):
    children = []
    for key in ("left", "right", "arg"):
        if isinstance(node.get(key), dict):
            children.append(node[key])
    children.extend(child for child in node.get("args", []) if isinstance(child, dict))
    child_metrics = [ast_complexity(child) for child in children]
    return {
        "node_count": 1 + sum(item["node_count"] for item in child_metrics),
        "max_depth": 1 + max([item["max_depth"] for item in child_metrics], default=0),
        "comparison_count": int(node["op"] in {"eq", "ne", "lt", "lte", "gt", "gte"})
            + sum(item["comparison_count"] for item in child_metrics),
        "boolean_operator_count": int(node["op"] in {"and", "or", "not"})
            + sum(item["boolean_operator_count"] for item in child_metrics),
        "field_reference_count": int(node["op"] == "field")
            + sum(item["field_reference_count"] for item in child_metrics),
    }

CASE_COMPLEXITY = {
    case_id: {
        **ast_complexity(case["predicate"]),
        "candidate_count": case["candidate_count"],
        "data_field_count": len(case["data"]),
        "diagnostic_dimension": case["diagnostic_dimension"],
        "unused_data_fields": [],
    }
    for case_id, case in PUBLIC_CASES.items()
}

for case_id, case in PUBLIC_CASES.items():
    n = case["candidate_count"]
    assert n == 8
    assert case["data"]
    assert all(len(values) == n for values in case["data"].values()), case_id

(OUTPUT_DIR / "case_complexity.json").write_text(
    json.dumps(CASE_COMPLEXITY, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Microcasos congelados:", list(PUBLIC_CASES))
display(pd.DataFrame([{
    "case_id": cid,
    "family": case["business_family"],
    "candidates": case["candidate_count"],
    "qdsv_route": case["qdsv_route"],
    **CASE_COMPLEXITY[cid],
} for cid, case in PUBLIC_CASES.items()]))


## 7. Ground truth congelado y físicamente separado

Las expectativas se derivan antes de invocar cualquier plataforma y nunca se entregan a los builders. Los dos casos de `compliance` son semánticamente equivalentes en el dominio binario congelado; los dos casos `cost <= ...` comparten resultados para aislar constante frente a campo.


In [ ]:
FROZEN_EXPECTATIONS = {
    "compliance_eq_constant_8": {"true_indices": [0, 1, 2, 4, 5, 6, 7]},
    "compliance_gte_constant_8": {"true_indices": [0, 1, 2, 4, 5, 6, 7]},
    "cost_eq_constant_8": {"true_indices": [0]},
    "cost_lte_constant_8": {"true_indices": [0, 1, 3, 4, 5, 6, 7]},
    "cost_lte_field_8": {"true_indices": [0, 1, 3, 4, 5, 6, 7]},
}

FORBIDDEN_KEYS = {
    "expected", "expected_true_indices", "true_indices", "answer", "answers",
    "solution", "winner", "winning_candidates", "mark_bit", "precomputed_result"
}

def forbidden_paths(obj, path="root"):
    hits=[]
    if isinstance(obj, dict):
        for key, value in obj.items():
            if str(key).lower() in FORBIDDEN_KEYS:
                hits.append(f"{path}.{key}")
            hits.extend(forbidden_paths(value, f"{path}.{key}"))
    elif isinstance(obj, (list, tuple)):
        for i, value in enumerate(obj):
            hits.extend(forbidden_paths(value, f"{path}[{i}]"))
    return hits

for case_id, case in PUBLIC_CASES.items():
    assert not forbidden_paths(case), (case_id, forbidden_paths(case))
    assert case_id in FROZEN_EXPECTATIONS

print("Separación aprobada: los builders reciben PUBLIC_CASES y nunca FROZEN_EXPECTATIONS.")


## 8. Auditoría independiente de los fixtures públicos


In [ ]:
def eval_public_ast(node, row):
    op=node["op"]
    if op=="field": return row[node["name"]]
    if op=="const": return node["value"]
    if op=="add": return sum(eval_public_ast(x,row) for x in node["args"])
    if op=="sub": return eval_public_ast(node["left"],row)-eval_public_ast(node["right"],row)
    if op=="mul": return eval_public_ast(node["left"],row)*eval_public_ast(node["right"],row)
    if op=="squared_diff":
        d=eval_public_ast(node["left"],row)-eval_public_ast(node["right"],row)
        return d*d
    if op in {"eq","ne","lt","lte","gt","gte"}:
        a,b=eval_public_ast(node["left"],row),eval_public_ast(node["right"],row)
        return {"eq":a==b,"ne":a!=b,"lt":a<b,"lte":a<=b,"gt":a>b,"gte":a>=b}[op]
    if op=="and": return all(bool(eval_public_ast(x,row)) for x in node["args"])
    if op=="or": return any(bool(eval_public_ast(x,row)) for x in node["args"])
    if op=="not": return not bool(eval_public_ast(node["arg"],row))
    raise ValueError(f"Operación pública desconocida: {op}")

def public_truth(case):
    result=[]
    for i in range(case["candidate_count"]):
        row={name:values[i] for name,values in case["data"].items()}
        if bool(eval_public_ast(case["predicate"],row)):
            result.append(i)
    return result

if VERIFY_FROZEN_EXPECTATIONS:
    for case_id, case in PUBLIC_CASES.items():
        observed=public_truth(case)
        frozen=FROZEN_EXPECTATIONS[case_id]["true_indices"]
        assert observed==frozen, (case_id, observed, frozen)
    print("Ground truth congelado coincide con los predicados públicos.")


## 9. Digests y artefactos de caso


In [ ]:
def canonical_json(obj):
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",",":"))

def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

CASE_DIGESTS = {cid: sha256_text(canonical_json(case)) for cid,case in PUBLIC_CASES.items()}
EXPECTATION_DIGESTS = {cid: sha256_text(canonical_json(exp)) for cid,exp in FROZEN_EXPECTATIONS.items()}

(OUTPUT_DIR / "public_cases.json").write_text(
    json.dumps(PUBLIC_CASES, ensure_ascii=False, indent=2), encoding="utf-8"
)
(OUTPUT_DIR / "frozen_expectations.json").write_text(
    json.dumps(FROZEN_EXPECTATIONS, ensure_ascii=False, indent=2), encoding="utf-8"
)
(OUTPUT_DIR / "case_digests.json").write_text(
    json.dumps({"cases":CASE_DIGESTS,"expectations":EXPECTATION_DIGESTS}, indent=2), encoding="utf-8"
)
print("Digests congelados.")


## 10. Replay ideal independiente, mapping estricto y timeout


In [ ]:
def expected_vector(expectation, n):
    true_set = set(expectation["true_indices"])
    return [i in true_set for i in range(n)]

def decode_register(index, positions):
    return sum(((index >> pos) & 1) << bit for bit, pos in enumerate(positions))

def register_positions(circuit):
    return {
        reg.name: [circuit.find_bit(q).index for q in reg]
        for reg in circuit.qregs
    }

EXACT_CANDIDATE_REGISTER_NAMES = (
    "candidate",
    "candidate_index",
    "index",
    "x",
    "qdsv_input",
)
EXACT_PREDICATE_REGISTER_NAMES = (
    "predicate",
    "decision",
    "qdsv_result",
    "result",
    "mark",
    "flag",
)

def validate_mapping(circuit, mapping, candidate_count):
    if not isinstance(mapping, dict):
        raise AdapterError("El mapping debe ser un diccionario explícito.")
    candidate_bits = list(mapping.get("candidate_bits") or [])
    predicate_bits = list(mapping.get("predicate_bits") or [])
    if not candidate_bits or not predicate_bits:
        raise AdapterError("El mapping requiere candidate_bits y predicate_bits.")
    if len(predicate_bits) != 1:
        raise AdapterError(f"El predicado debe ocupar exactamente un bit: {predicate_bits}")
    if len(set(candidate_bits)) != len(candidate_bits):
        raise AdapterError("candidate_bits contiene posiciones repetidas.")
    if len(set(predicate_bits)) != len(predicate_bits):
        raise AdapterError("predicate_bits contiene posiciones repetidas.")
    if set(candidate_bits) & set(predicate_bits):
        raise AdapterError("Los registros candidato y predicado se solapan.")
    all_bits = candidate_bits + predicate_bits
    if min(all_bits) < 0 or max(all_bits) >= circuit.num_qubits:
        raise AdapterError(
            f"Mapping fuera del circuito de {circuit.num_qubits} qubits: {all_bits}"
        )
    required_width = max(1, math.ceil(math.log2(candidate_count)))
    if len(candidate_bits) != required_width:
        raise AdapterError(
            f"Ancho candidato inesperado: {len(candidate_bits)}; esperado: {required_width}"
        )
    return {
        "candidate_bits": candidate_bits,
        "predicate_bits": predicate_bits,
        "source": mapping.get("source", "explicit"),
    }

def strict_mapping_from_exact_register_names(circuit, candidate_count):
    regs = register_positions(circuit)

    candidate_matches = [
        (name, regs[name])
        for name in EXACT_CANDIDATE_REGISTER_NAMES
        if name in regs
    ]
    predicate_matches = [
        (name, regs[name])
        for name in EXACT_PREDICATE_REGISTER_NAMES
        if name in regs
    ]

    if len(candidate_matches) != 1 or len(predicate_matches) != 1:
        raise KeyError(
            "Mapping no inequívoco por nombres exactos. "
            f"candidate_matches={candidate_matches}; "
            f"predicate_matches={predicate_matches}; registros={regs}"
        )

    candidate_name, candidate_bits = candidate_matches[0]
    predicate_name, predicate_bits = predicate_matches[0]
    return validate_mapping(
        circuit,
        {
            "candidate_bits": candidate_bits,
            "predicate_bits": predicate_bits,
            "source": f"exact_qreg_names:{candidate_name},{predicate_name}",
        },
        candidate_count,
    )

CONTRACT_CANDIDATE_KEYS = {
    "candidate_bits", "candidate_qubits", "candidate_positions", "index_bits"
}
CONTRACT_PREDICATE_KEYS = {
    "predicate_bits", "predicate_qubits", "result_bits", "decision_bits", "output_bits"
}

def _integer_positions(value):
    if isinstance(value, (list, tuple)) and value and all(
        isinstance(x, int) and not isinstance(x, bool) for x in value
    ):
        return list(value)
    return None

def mapping_from_artifact_contract(result, circuit, candidate_count):
    candidates = []

    def walk(obj, path="root"):
        if isinstance(obj, dict):
            measurements = obj.get("measurements")
            if isinstance(measurements, list):
                registers = register_positions(circuit)
                by_name = {
                    str(item.get("name")): str(item.get("register_name"))
                    for item in measurements
                    if isinstance(item, dict) and item.get("name") and item.get("register_name")
                }
                candidate_names = [name for name in ("x", "candidate", "candidate_index", "index") if name in by_name]
                predicate_names = [name for name in ("predicate", "decision", "result", "mark", "flag") if name in by_name]
                if len(candidate_names) == 1 and len(predicate_names) == 1:
                    candidate_register = by_name[candidate_names[0]]
                    predicate_register = by_name[predicate_names[0]]
                    if candidate_register in registers and predicate_register in registers:
                        candidates.append({
                            "candidate_bits": registers[candidate_register],
                            "predicate_bits": registers[predicate_register],
                            "source": f"measurement_contract:{path}",
                        })
            candidate_value = None
            predicate_value = None
            candidate_key = None
            predicate_key = None
            for key, value in obj.items():
                key_l = str(key).lower()
                if key_l in CONTRACT_CANDIDATE_KEYS:
                    positions = _integer_positions(value)
                    if positions is not None:
                        candidate_key, candidate_value = key_l, positions
                if key_l in CONTRACT_PREDICATE_KEYS:
                    positions = _integer_positions(value)
                    if positions is not None:
                        predicate_key, predicate_value = key_l, positions
            if candidate_value is not None and predicate_value is not None:
                candidates.append({
                    "candidate_bits": candidate_value,
                    "predicate_bits": predicate_value,
                    "source": f"artifact_contract:{path}:{candidate_key},{predicate_key}",
                })
            for key, value in obj.items():
                walk(value, f"{path}.{key}")
        elif isinstance(obj, (list, tuple)):
            for idx, value in enumerate(obj):
                walk(value, f"{path}[{idx}]")

    walk(result)
    valid = []
    errors = []
    for candidate in candidates:
        try:
            valid.append(validate_mapping(circuit, candidate, candidate_count))
        except Exception as exc:
            errors.append(str(exc))
    by_geometry = {}
    for item in valid:
        key = (tuple(item["candidate_bits"]), tuple(item["predicate_bits"]))
        entry = by_geometry.setdefault(key, {
            "candidate_bits": list(item["candidate_bits"]),
            "predicate_bits": list(item["predicate_bits"]),
            "sources": [],
        })
        source = item.get("source", "artifact_contract")
        if source not in entry["sources"]:
            entry["sources"].append(source)
    if len(by_geometry) == 1:
        resolved = next(iter(by_geometry.values()))
        resolved["source"] = "artifact_contract"
        return resolved
    if len(by_geometry) > 1:
        raise KeyError(
            "El contrato contiene geometrías de mapping incompatibles: "
            f"{list(by_geometry.values())}"
        )
    if errors:
        raise KeyError(f"Mappings de contrato inválidos: {errors}")
    return None

def resolve_mapping(circuit, candidate_count, explicit_hint=None, artifact_result=None):
    # Prioridad: contrato publico del artefacto, hint validado y nombres exactos.
    if artifact_result is not None:
        contract_mapping = mapping_from_artifact_contract(
            artifact_result, circuit, candidate_count
        )
        if contract_mapping is not None:
            return contract_mapping
    if explicit_hint:
        return validate_mapping(circuit, explicit_hint, candidate_count)
    return strict_mapping_from_exact_register_names(circuit, candidate_count)


def sampled_mps_replay(circuit, expected, mapping=None, artifact_result=None):
    from qiskit import ClassicalRegister

    qc = circuit.remove_final_measurements(inplace=False)
    try:
        resolved = resolve_mapping(
            qc,
            candidate_count=len(expected),
            explicit_hint=mapping,
            artifact_result=artifact_result,
        )
    except Exception as exc:
        return {
            "replay_status": "replay_mapping_required",
            "semantic_equivalence": None,
            "semantic_status": "not_evaluated",
            "reason": str(exc),
            "registers": register_positions(qc),
        }

    candidate_bits = resolved["candidate_bits"]
    predicate_bits = resolved["predicate_bits"]
    output = set(candidate_bits + predicate_bits)
    ancillas = [q for q in range(qc.num_qubits) if q not in output]
    ancilla_mask = sum(1 << q for q in ancillas)

    measured = qc.copy()
    classical = ClassicalRegister(qc.num_qubits, "measure_all")
    measured.add_register(classical)
    measured.measure(list(range(qc.num_qubits)), list(range(qc.num_qubits)))

    try:
        simulator = AerSimulator(method="matrix_product_state")
        executable = transpile(
            measured,
            simulator,
            optimization_level=0,
            seed_transpiler=SEED,
        )
        counts = simulator.run(
            executable,
            shots=SHOTS,
            seed_simulator=SEED,
        ).result().get_counts()
    except Exception as exc:
        return {
            "replay_status": "replay_error",
            "semantic_equivalence": None,
            "semantic_status": "not_evaluated",
            "reason": f"{type(exc).__name__}: {exc}",
        }

    mismatch = invalid = dirty = 0.0
    observed = {}
    candidate_marginals = {index: 0.0 for index in range(len(expected))}
    for bitstring, count in counts.items():
        probability = count / SHOTS
        basis_int = int(bitstring.replace(" ", ""), 2)
        candidate = decode_register(basis_int, candidate_bits)
        predicate = decode_register(basis_int, predicate_bits)
        observed[(candidate, predicate)] = observed.get((candidate, predicate), 0.0) + probability
        if candidate >= len(expected) or predicate not in (0, 1):
            invalid += probability
        else:
            candidate_marginals[candidate] += probability
            if bool(predicate) != bool(expected[candidate]):
                mismatch += probability
        if basis_int & ancilla_mask:
            dirty += probability

    candidate_count = len(expected)
    expected_marginal = 1 / candidate_count
    marginal_error = max(
        abs(value - expected_marginal)
        for value in candidate_marginals.values()
    )
    # Simultaneous Hoeffding bound across every candidate bin.
    distribution_tolerance = math.sqrt(
        math.log((2 * candidate_count) / CANDIDATE_DISTRIBUTION_ALPHA)
        / (2 * SHOTS)
    )
    distribution_consistent = marginal_error <= distribution_tolerance
    sampled_consistent = (
        mismatch == 0.0
        and invalid == 0.0
        and dirty == 0.0
        and distribution_consistent
    )
    pointwise_upper_95 = 1 - (0.05 ** (1 / SHOTS))
    familywise_upper_95 = 1 - (
        (ERROR_EVENT_FAMILY_ALPHA / ERROR_EVENT_COUNT) ** (1 / SHOTS)
    )

    return {
        "replay_status": "completed_sampled",
        "semantic_equivalence": None,
        "semantic_status": "sampled_consistent" if sampled_consistent else "sampled_mismatch",
        "sampled_consistency": sampled_consistent,
        "verification_strength": "sampled_mps",
        "shots": SHOTS,
        "observed_mismatch_rate": mismatch,
        "observed_invalid_rate": invalid,
        "observed_dirty_rate": dirty,
        "mismatch_probability": mismatch,
        "invalid_probability": invalid,
        "dirty_ancilla_probability": dirty,
        "mismatch_zero_event_upper_95_pointwise": pointwise_upper_95 if mismatch == 0.0 else None,
        "invalid_zero_event_upper_95_pointwise": pointwise_upper_95 if invalid == 0.0 else None,
        "dirty_zero_event_upper_95_pointwise": pointwise_upper_95 if dirty == 0.0 else None,
        "mismatch_zero_event_upper_95_familywise": familywise_upper_95 if mismatch == 0.0 else None,
        "invalid_zero_event_upper_95_familywise": familywise_upper_95 if invalid == 0.0 else None,
        "dirty_zero_event_upper_95_familywise": familywise_upper_95 if dirty == 0.0 else None,
        "candidate_distribution_status": (
            "statistically_consistent" if distribution_consistent else "statistically_inconsistent"
        ),
        "candidate_distribution_alpha": CANDIDATE_DISTRIBUTION_ALPHA,
        "candidate_distribution_tolerance": distribution_tolerance,
        "candidate_marginal_max_error": marginal_error,
        "candidate_bits": candidate_bits,
        "predicate_bits": predicate_bits,
        "mapping_source": resolved["source"],
        "mapping_sources": resolved.get("sources", [resolved["source"]]),
        "observed_distribution": {
            str(key): value for key, value in sorted(observed.items())
        },
    }


def exact_replay(circuit, expected, mapping=None, artifact_result=None):
    qc = circuit.remove_final_measurements(inplace=False)
    if qc.num_qubits > MAX_EXACT_QUBITS:
        return sampled_mps_replay(
            qc,
            expected,
            mapping=mapping,
            artifact_result=artifact_result,
        )

    try:
        resolved = resolve_mapping(
            qc,
            candidate_count=len(expected),
            explicit_hint=mapping,
            artifact_result=artifact_result,
        )
    except Exception as exc:
        return {
            "replay_status": "replay_mapping_required",
            "semantic_equivalence": None,
            "reason": str(exc),
            "registers": register_positions(qc),
        }

    candidate_bits = resolved["candidate_bits"]
    predicate_bits = resolved["predicate_bits"]

    try:
        simulator = AerSimulator(method="statevector")
        replay_circuit = transpile(
            qc,
            simulator,
            optimization_level=0,
            seed_transpiler=SEED,
        )
        if replay_circuit.num_qubits != qc.num_qubits:
            raise RuntimeError(
                f"El replay alteró el ancho lógico: {qc.num_qubits} -> {replay_circuit.num_qubits}"
            )
        replay_circuit.save_statevector()
        replay_result = simulator.run(replay_circuit).result()
        if not replay_result.success:
            raise RuntimeError(f"Aer no completó el replay: {replay_result.status}")
        statevector = np.asarray(
            replay_result.get_statevector(replay_circuit),
            dtype=complex,
        )
    except Exception as exc:
        return {
            "replay_status": "replay_error",
            "semantic_equivalence": None,
            "reason": f"{type(exc).__name__}: {exc}",
        }

    probs = np.abs(statevector) ** 2
    output = set(candidate_bits + predicate_bits)
    ancillas = [q for q in range(qc.num_qubits) if q not in output]
    ancilla_mask = sum(1 << q for q in ancillas)
    mismatch = invalid = dirty = 0.0
    observed = {}

    for basis in np.flatnonzero(probs > 1e-13):
        probability = float(probs[basis])
        basis_int = int(basis)
        candidate = decode_register(basis_int, candidate_bits)
        predicate = decode_register(basis_int, predicate_bits)
        observed[(candidate, predicate)] = (
            observed.get((candidate, predicate), 0.0) + probability
        )
        if candidate >= len(expected) or predicate not in (0, 1):
            invalid += probability
        elif bool(predicate) != bool(expected[candidate]):
            mismatch += probability
        if basis_int & ancilla_mask:
            dirty += probability

    equivalent = (
        mismatch <= 1e-10
        and invalid <= 1e-10
        and dirty <= 1e-10
    )
    return {
        "replay_status": "completed",
        "semantic_equivalence": equivalent,
        "verification_strength": "exact_statevector",
        "mismatch_probability": mismatch,
        "invalid_probability": invalid,
        "dirty_ancilla_probability": dirty,
        "candidate_bits": candidate_bits,
        "predicate_bits": predicate_bits,
        "mapping_source": resolved["source"],
        "mapping_sources": resolved.get("sources", [resolved["source"]]),
        "observed_distribution": {
            str(key): value for key, value in sorted(observed.items())
        },
    }

def load_qasm_source(source, fmt_hint=""):
    hint = str(fmt_hint).lower()
    if "openqasm 3" in source.lower() or "qasm3" in hint or hint.endswith("3"):
        return qasm3.loads(source), "qasm3"
    return qasm2.loads(source), "qasm2"

def extract_qdsv_artifact(result):
    artifact = result.get("artifact") or {}
    candidates = [
        artifact.get("content"),
        artifact.get("qasm"),
        result.get("qasm"),
        ((result.get("editable_artifacts") or {}).get("artifact_content")),
    ]
    source = next(
        (value for value in candidates if isinstance(value, str) and "OPENQASM" in value.upper()),
        None,
    )
    if source is None:
        raise ValueError("La respuesta QDSV no contiene OpenQASM en una ubicación reconocida")
    fmt = artifact.get("format") or result.get("artifact_type") or ""
    circuit, detected = load_qasm_source(source, fmt)
    return circuit, source, detected


def extract_explicit_qdsv_artifact(artifact):
    if not isinstance(artifact, dict):
        return None
    source = artifact.get("content")
    if not isinstance(source, str) or "OPENQASM" not in source.upper():
        return None
    circuit, detected = load_qasm_source(source, artifact.get("format") or "")
    return circuit, source, detected

def recursive_find_qasm(obj, seen=None):
    if seen is None:
        seen = set()
    oid = id(obj)
    if oid in seen:
        return None
    seen.add(oid)
    if isinstance(obj, str):
        if "OPENQASM" in obj.upper():
            return obj
        try:
            parsed = json.loads(obj)
        except Exception:
            return None
        return recursive_find_qasm(parsed, seen)
    if isinstance(obj, dict):
        for value in obj.values():
            found = recursive_find_qasm(value, seen)
            if found:
                return found
        return None
    if isinstance(obj, (list, tuple)):
        for value in obj:
            found = recursive_find_qasm(value, seen)
            if found:
                return found
        return None
    for method in ("model_dump", "dict"):
        fn = getattr(obj, method, None)
        if callable(fn):
            try:
                found = recursive_find_qasm(fn(), seen)
                if found:
                    return found
            except Exception:
                pass
    for method in ("model_dump_json", "json"):
        fn = getattr(obj, method, None)
        if callable(fn):
            try:
                found = recursive_find_qasm(fn(), seen)
                if found:
                    return found
            except Exception:
                pass
    if hasattr(obj, "__dict__"):
        return recursive_find_qasm(vars(obj), seen)
    return None


def _process_tree_rss_bytes(pid):
    try:
        root = psutil.Process(pid)
        processes = [root] + root.children(recursive=True)
    except psutil.Error:
        return 0
    total = 0
    for proc in processes:
        try:
            total += proc.memory_info().rss
        except psutil.Error:
            pass
    return total


def _terminate_process_tree(process):
    try:
        root = psutil.Process(process.pid)
        for child in root.children(recursive=True):
            try:
                child.kill()
            except psutil.Error:
                pass
    except psutil.Error:
        pass
    if process.is_alive():
        process.terminate()
        process.join(timeout=5)
    if process.is_alive():
        process.kill()
        process.join(timeout=5)


def run_process_limited(target, args, timeout_seconds, memory_limit_mb, stage):
    stage_started = time.monotonic()
    methods = mp.get_all_start_methods()
    if "fork" not in methods:
        raise RuntimeError(
            "Este notebook requiere Linux/Colab con multiprocessing 'fork'."
        )
    ctx = mp.get_context("fork")
    result_queue = ctx.Queue(maxsize=1)
    process = ctx.Process(target=target, args=(*args, result_queue))
    process.start()

    deadline = time.monotonic() + timeout_seconds
    memory_limit_bytes = int(memory_limit_mb * 1024 * 1024)
    payload = None
    peak_rss = 0
    limit_reason = None

    while time.monotonic() < deadline:
        try:
            payload = result_queue.get_nowait()
            break
        except queue_module.Empty:
            pass

        if not process.is_alive():
            try:
                payload = result_queue.get(timeout=1.0)
            except queue_module.Empty:
                payload = None
            break

        rss = _process_tree_rss_bytes(process.pid)
        peak_rss = max(peak_rss, rss)
        if memory_limit_bytes and rss > memory_limit_bytes:
            limit_reason = "memory_limit"
            break
        time.sleep(PROCESS_MONITOR_INTERVAL)

    if payload is None and limit_reason is None and process.is_alive():
        limit_reason = "timeout"

    if limit_reason is not None:
        _terminate_process_tree(process)
        return {
            "ok": False,
            "status": f"{stage}_{limit_reason}",
            "stage": stage,
            "error": (
                f"Etapa {stage} excedió {memory_limit_mb} MiB."
                if limit_reason == "memory_limit"
                else f"Etapa {stage} excedió {timeout_seconds} segundos."
            ),
            "peak_rss_mb": peak_rss / (1024 * 1024),
            "stage_seconds": time.monotonic() - stage_started,
        }

    process.join(timeout=5)
    if process.is_alive():
        _terminate_process_tree(process)
    if payload is None:
        return {
            "ok": False,
            "status": "worker_crash",
            "stage": stage,
            "error": f"Worker sin resultado; exitcode={process.exitcode}",
            "peak_rss_mb": peak_rss / (1024 * 1024),
            "stage_seconds": time.monotonic() - stage_started,
        }
    payload.setdefault("peak_rss_mb", peak_rss / (1024 * 1024))
    payload.setdefault("stage_seconds", time.monotonic() - stage_started)
    return payload


def _replay_worker(qasm_source, qasm_format, qpy_b64, expected, mapping_hint, artifact_result, result_queue):
    started = time.perf_counter()
    try:
        if qpy_b64:
            import base64 as _base64
            import io as _io
            from qiskit import qpy as _qpy
            circuits = _qpy.load(_io.BytesIO(_base64.b64decode(qpy_b64)))
            if len(circuits) != 1:
                raise ValueError(f"Se esperaba un circuito QPY; recibidos: {len(circuits)}")
            circuit = circuits[0]
            detected_format = "qpy"
        else:
            circuit, detected_format = load_qasm_source(qasm_source, qasm_format)

        replay = exact_replay(
            circuit,
            expected,
            mapping=mapping_hint,
            artifact_result=artifact_result,
        )
        result_queue.put({
            "ok": True,
            "status": replay.get("replay_status", "completed"),
            "stage": "replay",
            "detected_format": detected_format,
            "replay_seconds": time.perf_counter() - started,
            "native_qubits": circuit.num_qubits,
            "native_depth": circuit.depth(),
            "native_size": circuit.size(),
            "native_ops": {str(k): int(v) for k, v in circuit.count_ops().items()},
            **replay,
        })
    except MemoryError as exc:
        result_queue.put({
            "ok": False,
            "status": "replay_memory_limit",
            "stage": "replay",
            "error_type": type(exc).__name__,
            "error": str(exc),
            "traceback": traceback.format_exc(),
        })
    except Exception as exc:
        result_queue.put({
            "ok": False,
            "status": "replay_error",
            "stage": "replay",
            "error_type": type(exc).__name__,
            "error": str(exc),
            "traceback": traceback.format_exc(),
        })


def _normalization_worker(qasm_source, qasm_format, qpy_b64, result_queue):
    started = time.perf_counter()
    try:
        if qpy_b64:
            import base64 as _base64
            import io as _io
            from qiskit import qpy as _qpy
            circuits = _qpy.load(_io.BytesIO(_base64.b64decode(qpy_b64)))
            if len(circuits) != 1:
                raise ValueError(f'Se esperaba un circuito QPY; recibidos: {len(circuits)}')
            circuit = circuits[0]
        else:
            circuit, _ = load_qasm_source(qasm_source, qasm_format)
        base = circuit.remove_final_measurements(inplace=False)
        normalized = transpile(
            base,
            basis_gates=COMMON_BASIS,
            optimization_level=COMMON_OPT_LEVEL,
            seed_transpiler=SEED,
        )
        counts = normalized.count_ops()
        result_queue.put({
            "ok": True,
            "status": "normalized",
            "stage": "normalization",
            "normalized_qubits": normalized.num_qubits,
            "normalized_depth": normalized.depth(),
            "normalized_size": normalized.size(),
            "normalized_cx": int(counts.get("cx", 0)),
            "normalized_1q": int(sum(
                count for gate, count in counts.items()
                if gate in {"rz", "sx", "x"}
            )),
            "external_transpile_seconds": time.perf_counter() - started,
            "common_basis": " ".join(COMMON_BASIS),
            "common_optimization_level": COMMON_OPT_LEVEL,
            "seed_transpiler": SEED,
        })
    except MemoryError as exc:
        result_queue.put({
            "ok": False,
            "status": "normalization_memory_limit",
            "stage": "normalization",
            "error_type": type(exc).__name__,
            "error": str(exc),
            "traceback": traceback.format_exc(),
        })
    except Exception as exc:
        result_queue.put({
            "ok": False,
            "status": "normalization_error",
            "stage": "normalization",
            "error_type": type(exc).__name__,
            "error": str(exc),
            "traceback": traceback.format_exc(),
        })


## 10.1 Replay dinámico del quickstart de Bridge

In [ ]:
def _readme_smoke_worker(spec, result_queue):
    started = time.perf_counter()
    try:
        client = QDSVBridgeClient()
        result = client.generate(spec)
        recommended_role = result.get("recommended_artifact_role") or "canonical_ideal_artifact"
        recommended = (
            result.get("optimized_logical_artifact")
            if recommended_role == "optimized_logical_artifact"
            else result.get("artifact")
        ) or {}
        extracted = extract_explicit_qdsv_artifact(recommended)
        qasm_source = extracted[1] if extracted else None
        qasm_format = extracted[2] if extracted else recommended.get("format")
        mapping_hint = None
        if extracted:
            try:
                mapping_hint = strict_mapping_from_exact_register_names(
                    extracted[0], README_SMOKE_EXPECTATION["candidate_count"]
                )
            except Exception:
                pass
        delivery = result.get("artifact_delivery") or {}
        delivery_key = "optimized" if recommended_role == "optimized_logical_artifact" else "canonical"
        delivery_mode = (delivery.get(delivery_key) or {}).get("delivery_mode")
        result_queue.put({
            "ok": True,
            "status": "materialized_and_inline" if extracted else f"materialized_{delivery_mode or 'not_inline'}",
            "stage": "construction",
            "construction_seconds": time.perf_counter() - started,
            "result": json.loads(json.dumps(result, default=repr)),
            "qasm_source": qasm_source,
            "qasm_format": qasm_format,
            "mapping_hint": mapping_hint,
            "recommended_artifact_role": recommended_role,
            "recommended_delivery_mode": delivery_mode,
            "logical_optimization_status": (result.get("logical_optimization") or {}).get("status"),
        })
    except Exception as exc:
        result_queue.put({
            "ok": False,
            "status": "service_error",
            "stage": "construction",
            "error_type": type(exc).__name__,
            "error": str(exc),
            "traceback": traceback.format_exc(),
        })

README_SMOKE_RECORD = {
    "access_profile": ONBOARDING_ACCESS_PROFILE,
    "excluded_from_cross_platform_benchmark": True,
    "status": "not_run",
}

if RUN_README_SMOKE and QDSV_SERVICE_PREFLIGHT.get("ok"):
    construction_payload = run_process_limited(
        _readme_smoke_worker,
        (README_SMOKE_SPEC,),
        TIMEOUT_SECONDS["README_SMOKE_CONSTRUCTION"],
        MEMORY_LIMIT_MB["CONSTRUCTION"],
        "construction",
    )
    README_SMOKE_RECORD.update({
        key: value for key, value in construction_payload.items()
        if key not in {"result", "qasm_source", "traceback"}
    })
    if construction_payload.get("ok") and construction_payload.get("qasm_source"):
        smoke_result = construction_payload["result"]
        smoke_qasm = construction_payload["qasm_source"]
        replay_payload = run_process_limited(
            _replay_worker,
            (
                smoke_qasm,
                construction_payload["qasm_format"],
                None,
                expected_vector(
                    README_SMOKE_EXPECTATION,
                    README_SMOKE_EXPECTATION["candidate_count"],
                ),
                construction_payload.get("mapping_hint"),
                smoke_result,
            ),
            TIMEOUT_SECONDS["REPLAY"],
            MEMORY_LIMIT_MB["REPLAY"],
            "replay",
        )
        README_SMOKE_RECORD.update({
            key: value for key, value in replay_payload.items()
            if key not in {"traceback"}
        })
        if replay_payload.get("semantic_equivalence") is True:
            README_SMOKE_RECORD["status"] = "passed"
        elif replay_payload.get("semantic_equivalence") is False:
            README_SMOKE_RECORD["status"] = "semantic_mismatch"
        elif not replay_payload.get("ok"):
            README_SMOKE_RECORD["status"] = replay_payload.get("status", "replay_error")
        else:
            README_SMOKE_RECORD["status"] = replay_payload.get("replay_status", "incomplete")
        README_SMOKE_RECORD["artifact_sha256"] = hashlib.sha256(smoke_qasm.encode()).hexdigest()
        (OUTPUT_DIR / f"readme_smoke.{construction_payload['qasm_format']}").write_text(
            smoke_qasm, encoding="utf-8"
        )
        if replay_payload.get("traceback"):
            (OUTPUT_DIR / "traceback_readme_smoke_replay.txt").write_text(
                replay_payload["traceback"], encoding="utf-8"
            )
    elif construction_payload.get("ok"):
        README_SMOKE_RECORD["status"] = construction_payload.get("status")
        README_SMOKE_RECORD["external_replay_status"] = "not_available_without_inline_artifact"
    elif construction_payload.get("traceback"):
        (OUTPUT_DIR / "traceback_readme_smoke_construction.txt").write_text(
            construction_payload["traceback"], encoding="utf-8"
        )

elif RUN_README_SMOKE:
    README_SMOKE_RECORD.update({
        "status": "platform_preflight_blocked",
        "error_origin_stage": "service_preflight",
        "error": QDSV_SERVICE_PREFLIGHT.get("errors"),
    })

(OUTPUT_DIR / "readme_smoke_record.json").write_text(
    json.dumps(README_SMOKE_RECORD, ensure_ascii=False, indent=2, default=repr),
    encoding="utf-8",
)
display(pd.DataFrame([README_SMOKE_RECORD]))


## 11. Capa local QDSV Bridge - contrato público materializable de 0.6.1

La capa local `qdsv_spec_from_case()` convierte el predicado neutral a la especificación pública aceptada por `build_predicate_spec()`. No evalúa la regla, no incorpora ground truth y no genera el circuito. La materialización se delega a Bridge, que conserva el artefacto canónico como evidencia primaria.


In [ ]:
COMMON_ADAPTER_SOURCE = r"""
SUPPORTED_PUBLIC_AST_OPS = {
    "field", "const", "add", "sub", "mul", "squared_diff",
    "eq", "ne", "lt", "lte", "gt", "gte", "and", "or", "not",
}

def validate_public_ast(node, data_columns):
    if not isinstance(node, dict) or "op" not in node:
        raise AdapterError("Nodo AST inválido.")
    op = node["op"]
    if op not in SUPPORTED_PUBLIC_AST_OPS:
        raise AdapterNotImplemented(f"Operación pública no implementada por el adaptador: {op}")
    if op == "field":
        name = node.get("name")
        if name not in data_columns:
            raise AdapterError(f"Campo no disponible en el caso: {name}")
        return
    if op == "const":
        if not isinstance(node.get("value"), (int, float, bool)):
            raise AdapterError(f"Constante no numérica: {node.get('value')!r}")
        return
    for key in ("left", "right", "arg"):
        if key in node:
            validate_public_ast(node[key], data_columns)
    for child in node.get("args", []):
        validate_public_ast(child, data_columns)

def validate_public_case(case):
    required = {"title", "business_family", "candidate_count", "data", "predicate"}
    missing = required - set(case)
    if missing:
        raise AdapterError(f"Campos faltantes del caso: {sorted(missing)}")
    n = int(case["candidate_count"])
    if n <= 0:
        raise AdapterError("candidate_count debe ser positivo.")
    if not case["data"]:
        raise AdapterError("El caso no contiene métricas.")
    for name, values in case["data"].items():
        if len(values) != n:
            raise AdapterError(f"Longitud incorrecta para {name}: {len(values)} != {n}")
        if not all(isinstance(value, (int, float, bool)) for value in values):
            raise AdapterError(f"Valores no numéricos en {name}")
    validate_public_ast(case["predicate"], set(case["data"]))
"""
exec(COMMON_ADAPTER_SOURCE, globals())

QDSV_ADAPTER_SOURCE = r"""
from qdsv_bridge import build_predicate_spec

QDSV_NUMERIC_OPS = {"field", "const", "add", "sub", "mul", "squared_diff"}
QDSV_DECISIONS = {"eq", "ne", "lt", "lte", "gt", "gte"}


def ast_ops(node):
    result = {node["op"]}
    for key in ("left", "right", "arg"):
        if isinstance(node.get(key), dict):
            result |= ast_ops(node[key])
    for child in node.get("args", []):
        result |= ast_ops(child)
    return result


def qdsv_numeric_expr(node):
    op = node["op"]
    if op == "field":
        return {
            "op": "field",
            "dataset": "input_0",
            "row": {"var": "x"},
            "column": node["name"],
        }
    if op == "const":
        return node["value"]
    if op == "add":
        terms = [qdsv_numeric_expr(x) for x in node["args"]]
        if not terms:
            raise AdapterError("La suma publica requiere al menos un termino.")
        result = terms[0]
        for term in terms[1:]:
            result = {"op": "add", "args": [result, term]}
        return result
    if op in {"sub", "mul", "squared_diff"}:
        return {
            "op": op,
            "left": qdsv_numeric_expr(node["left"]),
            "right": qdsv_numeric_expr(node["right"]),
        }
    raise AdapterNotImplemented(f"Operacion QDSV no implementada en el adaptador: {op}")


def qdsv_spec_from_case(case):
    validate_public_case(case)
    predicate = case["predicate"]
    rows = [
        {
            "candidate_index": index,
            **{name: values[index] for name, values in case["data"].items()},
        }
        for index in range(case["candidate_count"])
    ]

    route = case.get("qdsv_route")
    if route == "bounded_predicate":
        spec = build_predicate_spec(
            rows=rows,
            predicate=predicate,
            shots=SHOTS,
            artifact_format="qasm2",
            backend_family="qiskit",
            materialization_mode="superposition_oracle",
            max_qubits=256,
            max_depth=1_000_000,
            logical_optimization=dict(LOGICAL_OPTIMIZATION_CONTRACT),
        )
        if forbidden_paths(spec):
            raise AdapterError(f"El spec contiene claves prohibidas: {forbidden_paths(spec)}")
        return spec
    if route != "score_expression":
        raise AdapterNotImplemented(f"Ruta QDSV publica no implementada: {route!r}")

    if predicate["op"] not in QDSV_DECISIONS:
        raise AdapterNotImplemented("La raiz del predicado no es una comparacion.")
    if predicate["right"].get("op") != "const":
        raise AdapterNotImplemented("El adaptador QDSV requiere un umbral constante.")
    numeric_ops = ast_ops(predicate["left"])
    if not numeric_ops <= QDSV_NUMERIC_OPS:
        raise AdapterNotImplemented(
            f"Operaciones numericas no implementadas: {sorted(numeric_ops-QDSV_NUMERIC_OPS)}"
        )

    term = {
        "name": "benchmark_public_expression",
        "value": qdsv_numeric_expr(predicate["left"]),
        "importance": 1,
        "priority": 1,
        "adjustments": [],
    }
    problem_spec = {
        "target": "quantum_hardware",
        "domain": {
            "variable": "x",
            "type": "int_range",
            "start": 0,
            "end": case["candidate_count"] - 1,
        },
        "data_binding": {
            "kind": "data_binding.v1",
            "datasets": [{
                "id": "input_0",
                "row_variable": "x",
                "index_field": "candidate_index",
                "rows": rows,
            }],
        },
        "model": {
            "kind": "score_model",
            "version": "2.0",
            "numeric_contract": {
                "output_scale": 8192,
                "max_function_states": 16384,
                "max_input_qubits": 20,
            },
            "score": {
                "terms": [term],
                "penalty": 0,
                "epsilon": 0.001,
                "decision": predicate["op"],
                "threshold": predicate["right"]["value"],
                "execution_strategy": "semantic_auto",
            },
        },
        "query": {"kind": "find_any"},
        "evidence": {"shots": SHOTS},
    }
    spec = {
        "problem_spec": problem_spec,
        "target": {
            "format": "qasm2",
            "backend_family": "qiskit",
            "logical_optimization": dict(LOGICAL_OPTIMIZATION_CONTRACT),
        },
        "limits": {"max_qubits": 256, "max_depth": 1_000_000},
        "materialization_policy": {
            "mode": "superposition_oracle",
            "shots": SHOTS,
        },
    }
    if forbidden_paths(spec):
        raise AdapterError(f"El spec contiene claves prohibidas: {forbidden_paths(spec)}")
    return spec


def classify_qdsv_exception(exc):
    payload = getattr(exc, "payload", None)
    text = (
        json.dumps(payload, default=repr)
        if payload is not None
        else str(exc)
    ).lower()
    if "quota" in text or "sdk_quota_exceeded" in text or "http 429" in text:
        return "quota_exhausted"
    if "resource" in text or "size_limit" in text or "max_qubit" in text or "max_depth" in text:
        return "resource_rejected"
    if "unsupported" in text or "missing_capabil" in text:
        return "unsupported"
    if "invalid" in text or "validation" in text:
        return "invalid_spec"
    if "timeout" in text:
        return "construction_timeout"
    if "connection" in text or "network" in text or "service" in text or "transport" in text:
        return "service_error"
    return "framework_construction_error"
"""
exec(QDSV_ADAPTER_SOURCE, globals())
print("Adaptador QDSV público ScoreModel + predicados compuestos cargado:", hashlib.sha256(QDSV_ADAPTER_SOURCE.encode()).hexdigest())


## 12. Implementación Qrisp 0.9.6 nativa e independiente

Cada rama expresa directamente el predicado con operadores públicos de `QuantumFloat` dentro de un `ConditionEnvironment`. No existe traducción AST, expansión ANF ni dependencia de código QDSV. La implementación es deliberadamente explícita porque Qrisp entrega una abstracción circuital, no una API de predicados empresariales equivalente a Bridge.


In [ ]:
QRISP_COMPATIBILITY_SOURCE = r"""

def activate_qrisp_packaging_compatibility():
    import hashlib as _hashlib
    import importlib as _importlib
    import importlib.metadata as _metadata
    import pathlib as _pathlib
    import sys as _sys

    expected_version = "0.9.6"
    observed_version = _metadata.version("qrisp")
    if observed_version != expected_version:
        raise AdapterError(
            f"Compatibilidad Qrisp limitada a {expected_version}; instalada: {observed_version}"
        )

    canonical_name = "qrisp.permeability.unqomp"
    packaged_name = "qrisp.permeability.qc_transformations.unqomp"
    applied = False
    try:
        module = _importlib.import_module(canonical_name)
        source_module = canonical_name
    except ModuleNotFoundError as exc:
        if exc.name != canonical_name:
            raise
        module = _importlib.import_module(packaged_name)
        _sys.modules[canonical_name] = module
        source_module = packaged_name
        applied = True

    module_path = _pathlib.Path(module.__file__).resolve()
    return {
        "version": "qrisp_packaging_compatibility.v1",
        "distribution": "qrisp",
        "distribution_version": observed_version,
        "canonical_module": canonical_name,
        "packaged_module": source_module,
        "alias_applied": applied,
        "module_path": str(module_path),
        "module_sha256": _hashlib.sha256(module_path.read_bytes()).hexdigest(),
        "framework_source_modified": False,
        "semantic_adapter_modified": False,
    }

"""
exec(QRISP_COMPATIBILITY_SOURCE, globals())

QRISP_ADAPTER_SOURCE = r"""

def qrisp_bit_width_unsigned(values):
    maximum = max([0, *[int(value) for value in values]])
    return max(1, math.ceil(math.log2(maximum + 1)))

def qrisp_validate_native_case(case_id, case):
    expected_fields = {
        "compliance_eq_constant_8": {"compliance"},
        "compliance_gte_constant_8": {"compliance"},
        "cost_eq_constant_8": {"cost"},
        "cost_lte_constant_8": {"cost"},
        "cost_lte_field_8": {"cost", "budget"},
    }
    if case_id not in expected_fields:
        raise AdapterNotImplemented(f"No existe builder Qrisp nativo para {case_id!r}.")
    if set(case) != {"candidate_count", "data"}:
        raise AdapterError("El builder Qrisp solo acepta candidate_count y data.")
    n = int(case["candidate_count"])
    if n <= 0:
        raise AdapterError("candidate_count debe ser positivo.")
    if set(case["data"]) != expected_fields[case_id]:
        raise AdapterError(
            f"Campos Qrisp inesperados para {case_id}: {sorted(case['data'])}"
        )
    for name, values in case["data"].items():
        if len(values) != n:
            raise AdapterError(f"Longitud incorrecta para {name}: {len(values)} != {n}")
        if not all(isinstance(value, (int, float, bool)) for value in values):
            raise AdapterError(f"Valores no numéricos en {name}")

def qrisp_prepare_index_exact(index, candidate_count):
    from qrisp import h, prepare
    padded_size = 1 << len(index.reg)
    if candidate_count == padded_size:
        h(index)
        return
    probabilities = [
        1 / candidate_count if value < candidate_count else 0.0
        for value in range(padded_size)
    ]
    prepare(index, np.sqrt(probabilities), method="qiskit")

def qrisp_toggle_explicit_field(index, field, values):
    from qrisp import control, x
    for candidate, value in enumerate(values):
        integer = int(value)
        for bit in range(len(field.reg)):
            if (integer >> bit) & 1:
                with control(index.reg, ctrl_state=candidate):
                    x(field[bit])

def qrisp_build_explicit_fields(index, case):
    from qrisp import QuantumFloat
    fields = {}
    loaders = []
    for name, values in case["data"].items():
        field = QuantumFloat(
            qrisp_bit_width_unsigned(values),
            signed=False,
            qs=index.qs,
            name=f"{name}_value",
        )
        qrisp_toggle_explicit_field(index, field, values)
        fields[name] = field
        loaders.append((field, list(values)))
    return fields, loaders

def qrisp_case_compliance_eq_constant(fields, decision, checkpoint):
    with fields["compliance"] == 1:
        checkpoint("native_condition_environment", current_native_event="condition_entered")
        decision.flip()

def qrisp_case_compliance_gte_constant(fields, decision, checkpoint):
    with fields["compliance"] >= 1:
        checkpoint("native_condition_environment", current_native_event="condition_entered")
        decision.flip()

def qrisp_case_cost_eq_constant(fields, decision, checkpoint):
    with fields["cost"] == 550:
        checkpoint("native_condition_environment", current_native_event="condition_entered")
        decision.flip()

def qrisp_case_cost_lte_constant(fields, decision, checkpoint):
    with fields["cost"] <= 600:
        checkpoint("native_condition_environment", current_native_event="condition_entered")
        decision.flip()

def qrisp_case_cost_lte_field(fields, decision, checkpoint):
    with fields["cost"] <= fields["budget"]:
        checkpoint("native_condition_environment", current_native_event="condition_entered")
        decision.flip()

QRISP_NATIVE_CASE_BUILDERS = {
    "compliance_eq_constant_8": qrisp_case_compliance_eq_constant,
    "compliance_gte_constant_8": qrisp_case_compliance_gte_constant,
    "cost_eq_constant_8": qrisp_case_cost_eq_constant,
    "cost_lte_constant_8": qrisp_case_cost_lte_constant,
    "cost_lte_field_8": qrisp_case_cost_lte_field,
}

def qrisp_apply_native_case(case_id, fields, decision, checkpoint):
    builder = QRISP_NATIVE_CASE_BUILDERS.get(case_id)
    if builder is None:
        raise AdapterNotImplemented(f"No existe builder Qrisp nativo para {case_id!r}.")
    checkpoint(
        "native_condition_environment",
        current_native_event="condition_environment_start",
        native_case_id=case_id,
    )
    builder(fields, decision, checkpoint)
    checkpoint(
        "field_cleanup",
        current_native_event="condition_environment_complete",
        native_condition_completed=True,
    )

def qrisp_clean_explicit_fields(index, loaders):
    for field, values in reversed(loaders):
        qrisp_toggle_explicit_field(index, field, values)
        field.delete(verify=False)

def qrisp_normalize_qubit_identifier(value):
    text = str(value).strip()
    for old, new in [(".", "_"), ("[", "_"), ("]", ""), (" ", "")]:
        text = text.replace(old, new)
    return text

def qrisp_qiskit_register_index_map(circuit):
    aliases = {}
    for qreg in circuit.qregs:
        for offset, bit in enumerate(qreg):
            index = circuit.find_bit(bit).index
            for name in {qreg.name, f"{qreg.name}_{offset}", f"{qreg.name}.{offset}"}:
                aliases.setdefault(qrisp_normalize_qubit_identifier(name), []).append(index)
    return aliases

def qrisp_locate_qubit_indices(compiled, qiskit_circuit, qubits):
    aliases = qrisp_qiskit_register_index_map(qiskit_circuit)
    indices = []
    sources = []
    for qubit in qubits:
        if qubit in compiled.qubits:
            indices.append(compiled.qubits.index(qubit))
            sources.append("compiled_qubit_identity")
            continue
        normalized = qrisp_normalize_qubit_identifier(qubit)
        matches = sorted(set(aliases.get(normalized, [])))
        if len(matches) != 1:
            raise AdapterError(
                f"No se pudo localizar inequívocamente {qubit!s}; matches={matches}"
            )
        indices.append(matches[0])
        sources.append("qiskit_register_fallback")
    source = sources[0] if len(set(sources)) == 1 else "mixed"
    return indices, source

def qrisp_write_phase_progress(progress_path, record):
    if not progress_path:
        return
    import pathlib as _pathlib
    path = _pathlib.Path(progress_path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(record, ensure_ascii=False, indent=2, default=repr),
        encoding="utf-8",
    )
    temporary.replace(path)

def build_qrisp_case(case_id, case, progress_path=None):
    qrisp_validate_native_case(case_id, case)
    compatibility_record = activate_qrisp_packaging_compatibility()

    import base64 as _base64
    import io as _io
    from qrisp import QuantumBool, QuantumFloat
    from qiskit import qpy as _qpy

    build_started = time.perf_counter()
    phase_timing = {
        "version": "qrisp_native_phase_timing.v1",
        "current_phase": "candidate_preparation",
        "current_native_event": "not_started",
        "completed": False,
    }

    def checkpoint(phase, **metrics):
        phase_timing.update(metrics)
        phase_timing["current_phase"] = phase
        phase_timing["elapsed_seconds_at_last_checkpoint"] = (
            time.perf_counter() - build_started
        )
        qrisp_write_phase_progress(progress_path, phase_timing)

    checkpoint("candidate_preparation")
    phase_started = time.perf_counter()
    index = QuantumFloat(
        max(1, math.ceil(math.log2(case["candidate_count"]))),
        signed=False,
        name="candidate",
    )
    qrisp_prepare_index_exact(index, case["candidate_count"])
    checkpoint(
        "field_loading",
        candidate_preparation_seconds=time.perf_counter() - phase_started,
    )

    phase_started = time.perf_counter()
    fields, loaders = qrisp_build_explicit_fields(index, case)
    checkpoint(
        "native_condition_environment",
        field_loading_seconds=time.perf_counter() - phase_started,
    )

    phase_started = time.perf_counter()
    decision = QuantumBool(qs=index.qs, name="decision")
    qrisp_apply_native_case(case_id, fields, decision, checkpoint)
    condition_environment_seconds = time.perf_counter() - phase_started

    phase_started = time.perf_counter()
    qrisp_clean_explicit_fields(index, loaders)
    checkpoint(
        "compile",
        condition_environment_seconds=condition_environment_seconds,
        field_cleanup_seconds=time.perf_counter() - phase_started,
    )

    phase_started = time.perf_counter()
    intended = list(index.reg) + list(decision.reg)
    compiled = decision.qs.compile(
        workspace=0,
        intended_measurements=intended,
        disable_uncomputation=False,
        compile_mcm=False,
    )
    checkpoint(
        "qiskit_conversion",
        compile_seconds=time.perf_counter() - phase_started,
    )

    phase_started = time.perf_counter()
    circuit = compiled.to_qiskit()
    checkpoint(
        "qpy_export",
        qiskit_conversion_seconds=time.perf_counter() - phase_started,
    )
    candidate_bits, candidate_mapping_source = qrisp_locate_qubit_indices(
        compiled, circuit, list(index.reg)
    )
    predicate_bits, predicate_mapping_source = qrisp_locate_qubit_indices(
        compiled, circuit, list(decision.reg)
    )
    mapping_hint = {
        "candidate_bits": candidate_bits,
        "predicate_bits": predicate_bits,
        "source": (
            candidate_mapping_source
            if candidate_mapping_source == predicate_mapping_source
            else "mixed"
        ),
    }

    phase_started = time.perf_counter()
    qpy_buffer = _io.BytesIO()
    _qpy.dump(circuit, qpy_buffer)
    qpy_b64 = _base64.b64encode(qpy_buffer.getvalue()).decode("ascii")
    checkpoint(
        "qasm_export",
        qpy_export_seconds=time.perf_counter() - phase_started,
    )

    phase_started = time.perf_counter()
    qasm_source = None
    qasm_export_status = "not_attempted"
    qasm_export_error = None
    try:
        qasm_source = qasm2.dumps(circuit)
        qasm_export_status = "exported"
    except Exception as exc:
        qasm_export_status = "optional_export_failed"
        qasm_export_error = f"{type(exc).__name__}: {exc}"

    checkpoint(
        "completed",
        qasm_export_seconds=time.perf_counter() - phase_started,
        total_build_seconds=time.perf_counter() - build_started,
        current_native_event="completed",
        completed=True,
    )

    return circuit, qasm_source, qpy_b64, {
        "qasm_export_status": qasm_export_status,
        "qasm_export_error": qasm_export_error,
        "mapping_hint": mapping_hint,
        "compatibility_record": compatibility_record,
        "strategy": "native_condition_environment",
        "native_route": case_id,
        "candidate_preparation": "exact_valid_domain",
        "workspace": 0,
        "disable_uncomputation": False,
        "intended_measurements": True,
        "observable_output": "decision",
        "phase_timing": phase_timing,
    }


"""
exec(QRISP_ADAPTER_SOURCE, globals())

QRISP_NATIVE_CASE_MANIFEST = {'version': 'qrisp_native_case_manifest.v1', 'framework': 'qrisp', 'framework_version': '0.9.6', 'strategy': 'native_condition_environment', 'uses_qdsv_code': False, 'uses_generic_ast_translation': False, 'uses_anf_expansion': False, 'uses_additional_auto_uncompute_wrapper': False, 'compile_disable_uncomputation': False, 'official_documentation': ['https://qrisp.eu/reference/Quantum%20Types/QuantumFloat.html', 'https://qrisp.eu/reference/Quantum%20Types/QuantumBool.html', 'https://qrisp.eu/reference/Quantum%20Environments/ConditionEnvironment.html'], 'cases': {'compliance_eq_constant_8': "with fields['compliance'] == 1", 'compliance_gte_constant_8': "with fields['compliance'] >= 1", 'cost_eq_constant_8': "with fields['cost'] == 550", 'cost_lte_constant_8': "with fields['cost'] <= 600", 'cost_lte_field_8': "with fields['cost'] <= fields['budget']"}}
(OUTPUT_DIR / "qrisp_native_case_manifest.json").write_text(
    json.dumps(QRISP_NATIVE_CASE_MANIFEST, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

QRISP_COMPATIBILITY_RECORD = activate_qrisp_packaging_compatibility()
ENVIRONMENT_SNAPSHOT["qrisp_packaging_compatibility"] = QRISP_COMPATIBILITY_RECORD
ENVIRONMENT_SNAPSHOT["qrisp_semantic_strategy"] = "native_condition_environment"
(OUTPUT_DIR / "environment.json").write_text(
    json.dumps(ENVIRONMENT_SNAPSHOT, ensure_ascii=False, indent=2), encoding="utf-8"
)
(OUTPUT_DIR / "qrisp_packaging_compatibility.json").write_text(
    json.dumps(QRISP_COMPATIBILITY_RECORD, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("Builder Qrisp nativo:", hashlib.sha256(QRISP_ADAPTER_SOURCE.encode()).hexdigest())
print("Estrategia Qrisp:", QRISP_NATIVE_CASE_MANIFEST["strategy"])
print("Casos Qrisp independientes:", list(QRISP_NATIVE_CASE_MANIFEST["cases"]))
print("Compatibilidad de empaquetado:", QRISP_COMPATIBILITY_RECORD)


## 13. Esfuerzo con código real y atribución separada

Las fuentes ejecutables se atribuyen sin combinar categorías incompatibles:

- `common_harness_loc`: validación neutral compartida; se informa una sola vez.
- `business_data_loc`: datos públicos comunes del problema.
- `neutral_predicate_specification_loc`: representación neutral del significado de la regla.
- `shared_platform_infrastructure_loc`: infraestructura reutilizable entre los casos actuales.
- `case_specific_platform_logic_loc`: código nuevo requerido por una regla concreta en esa plataforma.
- `end_user_invocation_loc`: llamada visible para someter el caso.
- `platform_specific_new_case_loc`: lógica específica más invocación; no incluye datos, predicado neutral ni infraestructura compartida.

QDSV consume la especificación neutral mediante su capa local reutilizable y Bridge. Qrisp reexpresa manualmente ese significado en una condición nativa por caso. No se calcula una métrica LOC única ni se presenta la infraestructura como código que todo usuario deba escribir.


In [ ]:
import ast as _ast

def nonempty_loc(source_text):
    return sum(
        1 for line in source_text.splitlines()
        if line.strip() and not line.lstrip().startswith("#")
    )

def top_level_function_source(source_text, function_name):
    tree = _ast.parse(source_text)
    lines = source_text.splitlines()
    for node in tree.body:
        if isinstance(node, (_ast.FunctionDef, _ast.AsyncFunctionDef)) and node.name == function_name:
            return "\n".join(lines[node.lineno - 1:node.end_lineno]) + "\n"
    raise AssertionError(f"Función no encontrada: {function_name}")

def render_business_data_literal(case_id, case):
    return (
        f"CASE_ID = {case_id!r}\n"
        f"CANDIDATE_COUNT = {case['candidate_count']!r}\n"
        f"BUSINESS_DATA = {pprint.pformat(case['data'], sort_dicts=False, width=100)}\n"
    )

def render_predicate_literal(case):
    return f"PUBLIC_PREDICATE = {pprint.pformat(case['predicate'], sort_dicts=False, width=100)}\n"

def render_case_literal(case_id, case):
    return (
        render_business_data_literal(case_id, case)
        + render_predicate_literal(case)
        + "BUSINESS_CASE = {\n"
        + "    'title': " + repr(case['title']) + ",\n"
        + "    'business_family': " + repr(case['business_family']) + ",\n"
        + "    'candidate_count': CANDIDATE_COUNT,\n"
        + "    'data': BUSINESS_DATA,\n"
        + "    'predicate': PUBLIC_PREDICATE,\n"
        + "    'qdsv_route': " + repr(case['qdsv_route']) + ",\n"
        + "}\n"
    )

def render_qdsv_invocation(case_id, case):
    qdsv_spec_from_case(case)
    return "\n".join([
        "from qdsv_bridge import QDSVBridgeClient, select_recommended_artifact",
        "spec = qdsv_spec_from_case(BUSINESS_CASE)",
        "client = QDSVBridgeClient()",
        "result = client.generate(spec)",
        "artifact = select_recommended_artifact(result)",
        "qasm_source = artifact['content']",
        "",
    ])

def render_qrisp_invocation(case_id, case):
    return "\n".join([
        "QRISP_INPUT = {\"candidate_count\": CANDIDATE_COUNT, \"data\": BUSINESS_DATA}",
        "circuit, qasm_source, qpy_b64, metadata = build_qrisp_case(CASE_ID, QRISP_INPUT)",
        "",
    ])

QRISP_CASE_FUNCTIONS = {
    "compliance_eq_constant_8": "qrisp_case_compliance_eq_constant",
    "compliance_gte_constant_8": "qrisp_case_compliance_gte_constant",
    "cost_eq_constant_8": "qrisp_case_cost_eq_constant",
    "cost_lte_constant_8": "qrisp_case_cost_lte_constant",
    "cost_lte_field_8": "qrisp_case_cost_lte_field",
}

COMMON_HARNESS_SOURCE = COMMON_ADAPTER_SOURCE
PLATFORM_ADAPTER_SOURCES = {"QDSV": QDSV_ADAPTER_SOURCE, "Qrisp": QRISP_ADAPTER_SOURCE}
ENVIRONMENT_COMPATIBILITY_SOURCES = {"QDSV": "", "Qrisp": QRISP_COMPATIBILITY_SOURCE}
AUTHENTICATION_INTEGRATION_SOURCES = {"QDSV": "", "Qrisp": ""}
CASE_RENDERERS = {"QDSV": render_qdsv_invocation, "Qrisp": render_qrisp_invocation}
MANUAL_RESPONSIBILITIES = {
    "QDSV": ["neutral_predicate_specification", "generate_call", "artifact_retrieval"],
    "Qrisp": [
        "register_allocation", "candidate_state_preparation", "explicit_reversible_field_loading",
        "native_case_condition", "condition_uncomputation", "compile_and_qasm_export",
    ],
}

QRISP_CASE_LOGIC_SOURCES = {
    case_id: top_level_function_source(QRISP_ADAPTER_SOURCE, function_name)
    for case_id, function_name in QRISP_CASE_FUNCTIONS.items()
}
QRISP_CASE_REGISTRATION_LOC = 1
QRISP_TOTAL_CASE_LOGIC_LOC = sum(
    nonempty_loc(text) + QRISP_CASE_REGISTRATION_LOC
    for text in QRISP_CASE_LOGIC_SOURCES.values()
)
SHARED_PLATFORM_INFRASTRUCTURE_LOC = {
    "QDSV": nonempty_loc(QDSV_ADAPTER_SOURCE),
    "Qrisp": nonempty_loc(QRISP_ADAPTER_SOURCE) - QRISP_TOTAL_CASE_LOGIC_LOC,
}

COMMON_HARNESS_LOC = nonempty_loc(COMMON_HARNESS_SOURCE)
(OUTPUT_DIR / "common_harness.py").write_text(COMMON_HARNESS_SOURCE, encoding="utf-8")

EFFORT_ROWS = []
for case_id, case in PUBLIC_CASES.items():
    business_data_source = render_business_data_literal(case_id, case)
    predicate_source = render_predicate_literal(case)
    case_definition_source = render_case_literal(case_id, case)
    business_data_loc = nonempty_loc(business_data_source)
    predicate_loc = nonempty_loc(predicate_source)
    case_definition_loc = nonempty_loc(case_definition_source)
    (OUTPUT_DIR / f"neutral_case_{case_id}.py").write_text(case_definition_source, encoding="utf-8")
    (OUTPUT_DIR / f"neutral_predicate_{case_id}.py").write_text(predicate_source, encoding="utf-8")

    for platform, renderer in CASE_RENDERERS.items():
        adapter_source = PLATFORM_ADAPTER_SOURCES[platform]
        if platform == "Qrisp":
            case_logic_source = QRISP_CASE_LOGIC_SOURCES[case_id]
            case_logic_loc = nonempty_loc(case_logic_source) + QRISP_CASE_REGISTRATION_LOC
            predicate_consumption = "manually_reexpressed_in_native_case_logic"
            (OUTPUT_DIR / f"platform_case_logic_qrisp_{case_id}.py").write_text(
                case_logic_source, encoding="utf-8"
            )
        else:
            case_logic_source = ""
            case_logic_loc = 0
            predicate_consumption = "structured_public_spec_consumed_by_reusable_layer"

        row = {
            "platform": platform,
            "case_id": case_id,
            "access_profile": PLATFORM_ACCESS_PROFILES[platform],
            "common_harness_loc": COMMON_HARNESS_LOC,
            "business_data_loc": business_data_loc,
            "neutral_predicate_specification_loc": predicate_loc,
            "neutral_case_definition_loc": case_definition_loc,
            "neutral_predicate_consumption": predicate_consumption,
            "shared_platform_infrastructure_loc": SHARED_PLATFORM_INFRASTRUCTURE_LOC[platform],
            "shared_platform_infrastructure_sha256": sha256_text(adapter_source),
            "case_specific_platform_logic_loc": case_logic_loc,
            "case_specific_platform_logic_sha256": sha256_text(case_logic_source),
            "environment_compatibility_loc": nonempty_loc(ENVIRONMENT_COMPATIBILITY_SOURCES[platform]),
            "environment_compatibility_sha256": sha256_text(ENVIRONMENT_COMPATIBILITY_SOURCES[platform]),
            "authentication_integration_loc": nonempty_loc(AUTHENTICATION_INTEGRATION_SOURCES[platform]),
            "manual_responsibilities": MANUAL_RESPONSIBILITIES[platform],
            "effort_status": "measured",
        }
        try:
            invocation_source = renderer(case_id, case)
            invocation_loc = nonempty_loc(invocation_source)
            row.update({
                "end_user_invocation_loc": invocation_loc,
                "platform_specific_new_case_loc": case_logic_loc + invocation_loc,
                "end_user_invocation_sha256": sha256_text(invocation_source),
            })
            (OUTPUT_DIR / f"platform_invocation_{platform.lower()}_{case_id}.py").write_text(
                invocation_source, encoding="utf-8"
            )
        except AdapterNotImplemented as exc:
            row.update({
                "effort_status": "adapter_not_implemented", "effort_error": str(exc),
                "end_user_invocation_loc": None, "platform_specific_new_case_loc": None,
                "end_user_invocation_sha256": None,
            })
        except Exception as exc:
            row.update({
                "effort_status": "adapter_error", "effort_error": f"{type(exc).__name__}: {exc}",
                "end_user_invocation_loc": None, "platform_specific_new_case_loc": None,
                "end_user_invocation_sha256": None,
            })
        EFFORT_ROWS.append(row)

for platform, adapter_source in PLATFORM_ADAPTER_SOURCES.items():
    (OUTPUT_DIR / f"platform_adapter_{platform.lower()}.py").write_text(adapter_source, encoding="utf-8")
    compatibility_source = ENVIRONMENT_COMPATIBILITY_SOURCES[platform]
    if compatibility_source:
        (OUTPUT_DIR / f"environment_compatibility_{platform.lower()}.py").write_text(
            compatibility_source, encoding="utf-8"
        )

LOC_ATTRIBUTION_POLICY = {
    "version": "loc_attribution_policy.v2",
    "common_harness": "reported_once_not_attributed_to_platform",
    "business_data": "shared_problem_input",
    "neutral_predicate_specification": "shared_semantic_meaning_executable_input_for_qdsv",
    "shared_platform_infrastructure": "reusable_across_current_cases",
    "case_specific_platform_logic": "new_platform_code_required_for_this_business_rule",
    "end_user_invocation": "visible_submission_surface",
    "single_combined_winner_metric": False,
}
(OUTPUT_DIR / "loc_attribution_policy.json").write_text(
    json.dumps(LOC_ATTRIBUTION_POLICY, ensure_ascii=False, indent=2), encoding="utf-8"
)

effort_df = pd.DataFrame(EFFORT_ROWS)
display(effort_df)
print("common_harness_loc (informado una sola vez):", COMMON_HARNESS_LOC)

PLATFORM_INDEPENDENCE_CONTRACT = {
    "version": "platform_independence_contract.v2",
    "claim": "independent_platform_specific_implementations_using_public_interfaces",
    "no_adapters_claim": False,
    "local_platform_glue_present": True,
    "qdsv_builder": "local_spec_layer_then_qdsv_bridge.build_predicate_spec",
    "qrisp_builder": "manual_native_condition_environment_per_case",
    "qrisp_track": "Qrisp 0.9.6 native public API + audited packaging compatibility shim",
    "cross_platform_imports": False,
    "cross_platform_artifact_reuse": False,
    "shared_inputs": ["public_business_data", "public_predicate_meaning"],
    "shared_postconstruction_harness": ["frozen_ground_truth", "mps_replay", "metrics", "normalization"],
    "ground_truth_visible_during_construction": False,
}
assert "qrisp" not in QDSV_ADAPTER_SOURCE.lower()
assert "qdsv" not in QRISP_ADAPTER_SOURCE.lower()
assert 'case["predicate"]' not in QRISP_ADAPTER_SOURCE
(OUTPUT_DIR / "platform_independence_contract.json").write_text(
    json.dumps(PLATFORM_INDEPENDENCE_CONTRACT, ensure_ascii=False, indent=2), encoding="utf-8"
)


## 14. Ejecucion independiente por etapa

Cada plataforma usa etapas aisladas de construccion, replay y normalizacion externa. Cada etapa tiene timeout y presupuesto de memoria propios.

En v19 se fuerza replay MPS muestreado para QDSV y Qrisp, independientemente de su numero de qubits, para mantener un evaluador comun. El resultado se clasifica como consistencia estadistica observada y nunca como equivalencia matematica.

`construction_seconds` representa latencia end-to-end observada: QDSV incluye la llamada y materialización remota, mientras Qrisp construye y compila localmente. No se interpreta como velocidad pura de compilador.


In [ ]:
RUN_ROWS = []
QDSV_VIEW_ROWS = []
NATIVE_ARTIFACTS = {}
NORMALIZATION_INPUTS = {}


def base_row(platform, case_id, case):
    return {
        "run_id": RUN_ID,
        "platform": platform,
        "case_id": case_id,
        "business_family": case["business_family"],
        "candidate_count": case["candidate_count"],
        "case_digest": CASE_DIGESTS[case_id],
        "access_profile": PLATFORM_ACCESS_PROFILES[platform],
        "status": "incomplete",
        "structural_status": "not_started",
        "semantic_equivalence": None,
        "primary_semantic_status": None,
        "common_case_eligible": False,
        "verification_strength": None,
    }


def replay_is_consistent(payload):
    return (
        payload.get('semantic_equivalence') is True
        or payload.get('semantic_status') == 'sampled_consistent'
    )


def replay_status_label(payload):
    if payload.get('semantic_equivalence') is True:
        return 'passed'
    return payload.get('semantic_status') or payload.get('replay_status', 'incomplete')


def save_trace(platform, case_id, stage, text):
    path = OUTPUT_DIR / f"traceback_{platform.lower()}_{case_id}_{stage}.txt"
    path.write_text(text or "", encoding="utf-8")
    return str(path)


def _serializable(value):
    return json.loads(json.dumps(value, default=repr))


def _qdsv_artifact_view(role, artifact, delivery):
    extracted = extract_explicit_qdsv_artifact(artifact)
    return {
        "role": role,
        "delivery_mode": delivery,
        "artifact_status": artifact.get("status") if isinstance(artifact, dict) else None,
        "artifact_digest": artifact.get("artifact_digest") if isinstance(artifact, dict) else None,
        "qasm_source": extracted[1] if extracted else None,
        "qasm_format": extracted[2] if extracted else None,
    }


def audit_qdsv_optimization_contract(result):
    errors = []
    raw_optimization = result.get("logical_optimization")
    target_contract = (result.get("target") or {}).get("logical_optimization")
    if not isinstance(raw_optimization, dict):
        digests = result.get("digests") or {}
        return {
            "status": "failed",
            "errors": ["requested_optimization_not_exposed"],
            "requested": dict(LOGICAL_OPTIMIZATION_CONTRACT),
            "observed_status": None,
            "engine_version": None,
            "canonical_artifact_digest": digests.get("artifact_digest"),
            "optimized_artifact_digest": None,
            "recommended_artifact_digest": digests.get("recommended_artifact_digest"),
        }
    optimization = raw_optimization
    expected = LOGICAL_OPTIMIZATION_CONTRACT
    if target_contract != expected:
        errors.append("requested_contract_not_echoed")
    if optimization.get("profile") != expected["profile"]:
        errors.append("profile_mismatch")
    if optimization.get("acceptance_policy") != expected["acceptance_policy"]:
        errors.append("acceptance_policy_mismatch")
    if optimization.get("engine") != "qiskit" or not optimization.get("engine_version"):
        errors.append("optimizer_engine_not_identified")
    for key in ("target", "backend", "coupling_map"):
        if optimization.get(key) is not None:
            errors.append(f"physical_{key}_unexpected")
    for key in ("layout", "routing", "scheduling", "approximation"):
        if optimization.get(key) is not False:
            errors.append(f"{key}_must_be_false")
    status = optimization.get("status")
    recommended_role = result.get("recommended_artifact_role")
    delivery = result.get("artifact_delivery") or {}
    digests = result.get("digests") or {}
    optimized = result.get("optimized_logical_artifact") or {}
    complete_evidence_statuses = {
        "accepted",
        "no_material_improvement",
        "rejected_semantic_validation",
        "rejected_contract_validation",
    }
    if status in complete_evidence_statuses:
        for key in ("resources_before", "resources_after", "digests", "validation", "acceptance"):
            if not isinstance(optimization.get(key), dict):
                errors.append(f"{key}_missing")

    if status == "accepted":
        if (optimization.get("validation") or {}).get("status") != "passed":
            errors.append("internal_validation_not_passed")
        if (optimization.get("acceptance") or {}).get("status") != "accepted":
            errors.append("pareto_acceptance_not_passed")
        if optimization.get("register_contract_preserved") is not True:
            errors.append("register_contract_not_preserved")
        if optimization.get("measurement_contract_preserved") is not True:
            errors.append("measurement_contract_not_preserved")
        if optimized.get("parent_digest") != digests.get("artifact_digest"):
            errors.append("optimized_parent_digest_mismatch")
        if not digests.get("optimized_artifact_digest"):
            errors.append("optimized_artifact_digest_missing")
        optimized_delivery = (delivery.get("optimized") or {}).get("delivery_mode")
        expected_role = (
            "optimized_logical_artifact"
            if optimized_delivery == "inline"
            else "canonical_ideal_artifact"
        )
        if recommended_role != expected_role:
            errors.append("recommended_role_incoherent")
    elif status == "no_material_improvement":
        if recommended_role != "canonical_ideal_artifact":
            errors.append("canonical_not_recommended_without_accepted_optimization")
    elif status in {"resource_skipped", "optimization_error"}:
        if recommended_role != "canonical_ideal_artifact":
            errors.append("canonical_not_recommended_after_incomplete_optimization")
        if optimized.get("status") != status:
            errors.append("optimized_artifact_status_mismatch")
        if digests.get("optimized_artifact_digest") is not None:
            errors.append("unexpected_optimized_artifact_digest")
    else:
        errors.append(f"unexpected_optimization_status:{status}")

    if errors:
        contract_status = "failed"
    elif status in {"accepted", "no_material_improvement"}:
        contract_status = "passed"
    else:
        contract_status = "incomplete"
    return {
        "status": contract_status,
        "errors": errors,
        "requested": dict(expected),
        "observed_status": status,
        "observed_reason": optimization.get("reason"),
        "engine_version": optimization.get("engine_version"),
        "canonical_artifact_digest": digests.get("artifact_digest"),
        "optimized_artifact_digest": digests.get("optimized_artifact_digest"),
        "recommended_artifact_digest": digests.get("recommended_artifact_digest"),
    }


def _platform_case_worker(platform, case_id, case, result_queue):
    started = time.perf_counter()
    stage = "adapter_validation"
    try:
        validate_public_case(case)
        if platform == "QDSV":
            stage = "adapter_translation"
            spec = qdsv_spec_from_case(case)
            stage = "qdsv_generate"
            client = QDSVBridgeClient()
            remote_started = time.perf_counter()
            result = client.generate(spec)
            remote_elapsed = time.perf_counter() - remote_started
            stage = "artifact_export"
            recommended_role = result.get("recommended_artifact_role") or "canonical_ideal_artifact"
            primary_role = "canonical_ideal_artifact"
            canonical = result.get("artifact") or {}
            extracted = extract_explicit_qdsv_artifact(canonical)
            qasm_source = extracted[1] if extracted else None
            qasm_format = extracted[2] if extracted else canonical.get("format")
            mapping_hint = None
            if extracted:
                try:
                    mapping_hint = strict_mapping_from_exact_register_names(
                        extracted[0], case["candidate_count"]
                    )
                except Exception:
                    pass
            delivery = result.get("artifact_delivery") or {}
            primary_delivery = (delivery.get("canonical") or {}).get("delivery_mode")
            recommended_delivery_key = (
                "optimized" if recommended_role == "optimized_logical_artifact" else "canonical"
            )
            recommended_delivery = (
                delivery.get(recommended_delivery_key) or {}
            ).get("delivery_mode")
            views = [
                _qdsv_artifact_view(
                    "canonical_ideal_artifact",
                    result.get("artifact") or {},
                    (delivery.get("canonical") or {}).get("delivery_mode"),
                ),
                _qdsv_artifact_view(
                    "optimized_logical_artifact",
                    result.get("optimized_logical_artifact") or {},
                    (delivery.get("optimized") or {}).get("delivery_mode"),
                ),
            ]
            optimization_audit = audit_qdsv_optimization_contract(result)
            result_queue.put({
                "ok": True,
                "status": "materialized_and_inline" if extracted else f"materialized_{primary_delivery or 'not_inline'}",
                "stage": "construction",
                "qasm_source": qasm_source, "qasm_format": qasm_format,
                "mapping_hint": mapping_hint,
                "platform_result": _serializable(result),
                "artifact_views": views,
                "primary_artifact_role": primary_role,
                "primary_delivery_mode": primary_delivery,
                "recommended_artifact_role": recommended_role,
                "recommended_delivery_mode": recommended_delivery,
                "logical_optimization": _serializable(result.get("logical_optimization")),
                "optimization_contract_audit": optimization_audit,
                "remote_materialization_seconds": remote_elapsed,
                "construction_seconds": time.perf_counter() - started,
            })
            return

        if platform == "Qrisp":
            stage = "framework_construction"
            build_started = time.perf_counter()
            progress_path = OUTPUT_DIR / f"qrisp_{case_id}_phase_progress.json"
            circuit, qasm_source, qpy_b64, metadata = build_qrisp_case(
                case_id,
                {
                    "candidate_count": case["candidate_count"],
                    "data": {name: list(values) for name, values in case["data"].items()},
                },
                progress_path=str(progress_path),
            )
            result_queue.put({
                "ok": True, "status": "materialized", "stage": "construction",
                "qasm_source": qasm_source, "qasm_format": "qasm2",
                "qpy_b64": qpy_b64,
                "mapping_hint": metadata.get("mapping_hint"),
                "compatibility_record": metadata.get("compatibility_record"),
                "qrisp_strategy": metadata.get("strategy"),
                "qrisp_candidate_preparation": metadata.get("candidate_preparation"),
                "qrisp_native_route": metadata.get("native_route"),
                "qrisp_workspace": metadata.get("workspace"),
                "qrisp_disable_uncomputation": metadata.get("disable_uncomputation"),
                "qrisp_phase_timing": metadata.get("phase_timing"),
                "qasm_export_status": metadata.get("qasm_export_status"),
                "qasm_export_error": metadata.get("qasm_export_error"),
                "local_compile_seconds": time.perf_counter() - build_started,
                "construction_seconds": time.perf_counter() - started,
            })
            return

        raise AdapterError(f"Plataforma desconocida: {platform}")

    except AdapterNotImplemented as exc:
        failure = {
            "status": "adapter_not_implemented",
            "error_type": type(exc).__name__,
            "error": str(exc),
            "traceback": traceback.format_exc(),
        }
    except AdapterError as exc:
        failure = {
            "status": "adapter_error",
            "error_type": type(exc).__name__,
            "error": str(exc),
            "traceback": traceback.format_exc(),
        }
    except Exception as exc:
        trace = traceback.format_exc()
        text = (str(exc) + "\n" + trace).lower()
        if stage == "qdsv_generate":
            status = classify_qdsv_exception(exc)
        elif platform == "Qrisp" and isinstance(exc, ModuleNotFoundError):
            status = "framework_installation_error"
        elif stage in {"adapter_validation", "adapter_translation", "adapter_source_generation"}:
            status = "adapter_error"
        elif stage == "artifact_export":
            status = "artifact_export_error"
        else:
            status = "framework_construction_error"
        failure = {
            "status": status,
            "error_type": type(exc).__name__,
            "error": str(exc),
            "traceback": trace,
        }
    result_queue.put({
        "ok": False,
        "stage": stage,
        "construction_seconds": time.perf_counter() - started,
        **failure,
    })



PLATFORM_ENABLED = {"QDSV": RUN_QDSV, "Qrisp": RUN_QRISP}
PLATFORM_PREFLIGHT_RECORDS = {
    "QDSV": {
        "canonical": dict(QDSV_CANONICAL_PREFLIGHT),
        "optimization_optional": dict(QDSV_OPTIMIZATION_PREFLIGHT),
    },
    "Qrisp": {
        "version": "qrisp_adapter_qualification_reference.v1",
        "status": "not_required",
        "strategy": "native_condition_environment",
        "qualification": "official_public_api_pattern",
        "battery_policy": "execute_each_case_independently",
    },
}

for platform in ("QDSV", "Qrisp"):
    case_order = list(PUBLIC_CASES)
    platform_block = None
    if platform == "QDSV" and not QDSV_SERVICE_PREFLIGHT.get("ok"):
        platform_block = dict(QDSV_SERVICE_PREFLIGHT)

    for case_id in case_order:
        case = PUBLIC_CASES[case_id]
        row = base_row(platform, case_id, case)
        if platform_block is not None:
            row.update({
                "status": "platform_preflight_blocked",
                "structural_status": "not_run",
                "error_origin_stage": "platform_preflight",
                "preflight_source_status": platform_block.get("status"),
                "error": platform_block.get("errors") or platform_block.get("error"),
            })
            RUN_ROWS.append(row)
            continue
        if not PLATFORM_ENABLED[platform]:
            row.update({"status": "incomplete", "structural_status": "not_run"})
            RUN_ROWS.append(row)
            continue

        construction_timeout = TIMEOUT_SECONDS[f"{platform}_CONSTRUCTION"]
        construction = run_process_limited(
            _platform_case_worker,
            (platform, case_id, case),
            construction_timeout,
            MEMORY_LIMIT_MB["CONSTRUCTION"],
            "construction",
        )
        row.update({
            "construction_timeout_budget_seconds": construction_timeout,
            "construction_memory_budget_mb": MEMORY_LIMIT_MB["CONSTRUCTION"],
            "construction_peak_rss_mb": construction.get("peak_rss_mb"),
            "memory_metric_scope": "absolute_process_tree_rss_including_inherited_pages",
            "memory_metric_comparable_across_platforms": False,
            "construction_seconds": (
                construction.get("construction_seconds")
                or construction.get("stage_seconds")
            ),
        })
        if platform == "Qrisp":
            progress_file = OUTPUT_DIR / f"qrisp_{case_id}_phase_progress.json"
            if progress_file.exists():
                phase_timing = json.loads(progress_file.read_text(encoding="utf-8"))
                row["qrisp_phase_timing"] = phase_timing
                for timing_key in (
                    "candidate_preparation_seconds",
                    "field_loading_seconds",
                    "condition_environment_seconds",
                    "field_cleanup_seconds",
                    "compile_seconds",
                    "qiskit_conversion_seconds",
                    "qpy_export_seconds",
                    "qasm_export_seconds",
                    "total_build_seconds",
                    "elapsed_seconds_at_last_checkpoint",
                    "current_phase",
                    "current_native_event",
                ):
                    row[f"qrisp_{timing_key}"] = phase_timing.get(timing_key)
        if not construction.get("ok"):
            row.update({
                "status": construction.get("status", "worker_crash"),
                "structural_status": "not_submitted" if construction.get("status") in {"adapter_error", "adapter_not_implemented"} else "error",
                "error_origin_stage": construction.get("stage"),
                "error_type": construction.get("error_type"),
                "error": construction.get("error"),
            })
            if construction.get("traceback"):
                row["traceback_file"] = save_trace(platform, case_id, "construction", construction["traceback"])
            RUN_ROWS.append(row)
            continue


        qasm_source = construction.get("qasm_source")
        qasm_format = construction.get("qasm_format")
        qpy_b64 = construction.get("qpy_b64")
        platform_result = construction.get("platform_result")
        primary_role = construction.get("primary_artifact_role") if platform == "QDSV" else "platform_native_artifact"
        recommended_role = construction.get("recommended_artifact_role") if platform == "QDSV" else primary_role
        if qasm_source or qpy_b64:
            replay = run_process_limited(
                _replay_worker,
                (
                    qasm_source,
                    qasm_format,
                    qpy_b64,
                    expected_vector(FROZEN_EXPECTATIONS[case_id], case["candidate_count"]),
                    REGISTER_HINTS.get((platform, case_id)) or construction.get("mapping_hint"),
                    platform_result if platform == "QDSV" else None,
                ),
                TIMEOUT_SECONDS["REPLAY"],
                MEMORY_LIMIT_MB["REPLAY"],
                "replay",
            )
        else:
            replay = {
                "ok": False,
                "status": "replay_not_available",
                "stage": "replay",
                "reason": "no_replayable_qasm_or_qpy_artifact",
                "semantic_equivalence": None,
            }
        qpy_raw = None
        if qpy_b64:
            import base64 as _base64
            qpy_raw = _base64.b64decode(qpy_b64)
        primary_artifact_bytes = (
            len(qasm_source.encode()) if qasm_source else len(qpy_raw) if qpy_raw else None
        )
        primary_artifact_sha256 = (
            sha256_text(qasm_source)
            if qasm_source
            else hashlib.sha256(qpy_raw).hexdigest() if qpy_raw else None
        )
        optimization_audit = construction.get("optimization_contract_audit") or {}
        row.update({
            "structural_status": "materialized",
            "native_artifact_role": primary_role,
            "recommended_artifact_role": recommended_role,
            "primary_delivery_mode": construction.get("primary_delivery_mode"),
            "recommended_delivery_mode": construction.get("recommended_delivery_mode"),
            "artifact_format": qasm_format if qasm_source else "qpy" if qpy_raw else None,
            "artifact_bytes": primary_artifact_bytes,
            "artifact_sha256": primary_artifact_sha256,
            "qasm_export_status": construction.get("qasm_export_status"),
            "qasm_export_error": construction.get("qasm_export_error"),
            "qpy_bytes": len(qpy_raw) if qpy_raw else None,
            "qpy_sha256": hashlib.sha256(qpy_raw).hexdigest() if qpy_raw else None,
            "remote_materialization_seconds": construction.get("remote_materialization_seconds"),
            "local_compile_seconds": construction.get("local_compile_seconds"),
            "environment_compatibility": construction.get("compatibility_record"),
            "qrisp_strategy": construction.get("qrisp_strategy"),
            "qrisp_candidate_preparation": construction.get("qrisp_candidate_preparation"),
            "qrisp_native_route": construction.get("qrisp_native_route"),
            "qrisp_workspace": construction.get("qrisp_workspace"),
            "qrisp_disable_uncomputation": construction.get("qrisp_disable_uncomputation"),
            "logical_optimization_status": (construction.get("logical_optimization") or {}).get("status"),
            "optimization_contract_status": optimization_audit.get("status") if platform == "QDSV" else None,
            "optimization_contract_errors": optimization_audit.get("errors") if platform == "QDSV" else None,
            "optimizer_qiskit_version": optimization_audit.get("engine_version") if platform == "QDSV" else None,
            "replay_timeout_budget_seconds": TIMEOUT_SECONDS["REPLAY"],
            "replay_memory_budget_mb": MEMORY_LIMIT_MB["REPLAY"],
            "replay_peak_rss_mb": replay.get("peak_rss_mb"),
        })
        if replay.get("ok"):
            row.update({key: value for key, value in replay.items() if key not in {"ok", "traceback"}})
            row["status"] = replay_status_label(replay)
            row["primary_semantic_status"] = replay_status_label(replay)
            row["common_case_eligible"] = replay_is_consistent(replay)
        else:
            row.update({
                "status": replay.get("status", "replay_error"),
                "error_origin_stage": replay.get("stage", "replay"),
                "error_type": replay.get("error_type"),
                "error": replay.get("error"),
            })
            if replay.get("traceback"):
                row["replay_traceback_file"] = save_trace(platform, case_id, "replay", replay["traceback"])

        if platform != "QDSV" and replay_is_consistent(replay):
            NORMALIZATION_INPUTS[(platform, case_id, "platform_native_artifact")] = (qasm_source, qasm_format, qpy_b64)
        if qasm_source:
            NATIVE_ARTIFACTS[(platform, case_id)] = qasm_source
            (OUTPUT_DIR / f"{platform.lower()}_{case_id}_recommended.{qasm_format}").write_text(qasm_source, encoding="utf-8")
        if qpy_b64 and platform == "Qrisp":
            import base64 as _base64
            (OUTPUT_DIR / f"{platform.lower()}_{case_id}_native.qpy").write_bytes(
                _base64.b64decode(qpy_b64)
            )
        if platform_result is not None:
            (OUTPUT_DIR / f"qdsv_{case_id}.json").write_text(
                json.dumps(platform_result, ensure_ascii=False, indent=2, default=repr), encoding="utf-8"
            )
            (OUTPUT_DIR / f"qdsv_optimization_contract_{case_id}.json").write_text(
                json.dumps(optimization_audit, ensure_ascii=False, indent=2, default=repr), encoding="utf-8"
            )
            view_statuses = {}
            view_artifacts = {}
            for view in construction.get("artifact_views") or []:
                role = view.get("role")
                view_source = view.get("qasm_source")
                view_format = view.get("qasm_format")
                view_row = {
                    "run_id": RUN_ID,
                    "platform": "QDSV",
                    "case_id": case_id,
                    "artifact_role": role,
                    "recommended": role == recommended_role,
                    "delivery_mode": view.get("delivery_mode"),
                    "artifact_status": view.get("artifact_status"),
                    "artifact_digest": (
                        optimization_audit.get("canonical_artifact_digest")
                        if role == "canonical_ideal_artifact"
                        else optimization_audit.get("optimized_artifact_digest")
                    ),
                    "semantic_equivalence": None,
                }
                if not view_source:
                    view_row["status"] = "metadata_only" if view.get("delivery_mode") == "metadata_only" else "not_available"
                    view_statuses[role] = view_row["status"]
                    QDSV_VIEW_ROWS.append(view_row)
                    continue
                if role == primary_role:
                    view_replay = replay
                else:
                    view_replay = run_process_limited(
                        _replay_worker,
                        (
                            view_source,
                            view_format,
                            None,
                            expected_vector(FROZEN_EXPECTATIONS[case_id], case["candidate_count"]),
                            construction.get("mapping_hint"),
                            platform_result,
                        ),
                        TIMEOUT_SECONDS["REPLAY"],
                        MEMORY_LIMIT_MB["REPLAY"],
                        "replay",
                    )
                view_row.update({
                    "artifact_format": view_format,
                    "artifact_bytes": len(view_source.encode()),
                    "artifact_sha256": sha256_text(view_source),
                    **{key: value for key, value in view_replay.items() if key not in {"ok", "traceback"}},
                })
                view_row["status"] = replay_status_label(view_replay)
                view_statuses[role] = view_row["status"]
                view_artifacts[role] = (view_source, view_format)
                (OUTPUT_DIR / f"qdsv_{case_id}_{role}.{view_format}").write_text(view_source, encoding="utf-8")
                QDSV_VIEW_ROWS.append(view_row)

            canonical_status = view_statuses.get("canonical_ideal_artifact", "not_available")
            optimized_status = view_statuses.get("optimized_logical_artifact", "not_available")
            recommended_status = view_statuses.get(recommended_role, "not_available")
            canonical_consistent = canonical_status in {"passed", "sampled_consistent"}
            recommended_consistent = recommended_status in {"passed", "sampled_consistent"}
            row.update({
                "canonical_semantic_status": canonical_status,
                "optimized_semantic_status": optimized_status,
                "recommended_semantic_status": recommended_status,
                "primary_semantic_status": canonical_status,
                "semantic_equivalence": True if canonical_status == "passed" else None,
                "common_case_eligible": canonical_consistent,
                "status": canonical_status,
            })

            canonical_artifact = view_artifacts.get("canonical_ideal_artifact")
            recommended_artifact = view_artifacts.get(recommended_role)
            if canonical_consistent and canonical_artifact:
                NORMALIZATION_INPUTS[("QDSV", case_id, "canonical_ideal_artifact")] = canonical_artifact
            if recommended_consistent and recommended_artifact:
                NORMALIZATION_INPUTS[("QDSV", case_id, "recommended_artifact")] = recommended_artifact
        RUN_ROWS.append(row)

(OUTPUT_DIR / "platform_preflight.json").write_text(
    json.dumps(PLATFORM_PREFLIGHT_RECORDS, ensure_ascii=False, indent=2, default=repr),
    encoding="utf-8",
)

results_df = pd.DataFrame(RUN_ROWS)
qdsv_view_columns = [
    "run_id", "platform", "case_id", "artifact_role", "recommended",
    "delivery_mode", "artifact_status", "artifact_digest",
    "semantic_equivalence", "semantic_status", "verification_strength",
    "native_qubits", "native_depth", "native_size", "status",
]
qdsv_views_df = pd.DataFrame(QDSV_VIEW_ROWS)
if qdsv_views_df.empty:
    qdsv_views_df = pd.DataFrame(columns=qdsv_view_columns)
display(results_df)
print("Vistas nativas QDSV: canonica y optimizada/recomendada")
display(qdsv_views_df)


## 15. Vista nativa y normalización externa común aislada

La vista nativa usa métricas obtenidas durante el replay protegido. La normalización externa se ejecuta en otro proceso con timeout y memoria propios; nunca modifica ni sustituye el artefacto nativo.


In [ ]:
NORMALIZED_ROWS = []
for (platform, case_id, input_artifact_role), artifact_payload in NORMALIZATION_INPUTS.items():
    qasm_source, qasm_format, *extra = artifact_payload
    qpy_b64 = extra[0] if extra else None
    payload = run_process_limited(
        _normalization_worker,
        (qasm_source, qasm_format, qpy_b64),
        TIMEOUT_SECONDS["NORMALIZATION"],
        MEMORY_LIMIT_MB["NORMALIZATION"],
        "normalization",
    )
    row = {
        "run_id": RUN_ID,
        "platform": platform,
        "case_id": case_id,
        "input_artifact_role": input_artifact_role,
        "comparison_track": (
            "neutral_primary"
            if input_artifact_role in {"canonical_ideal_artifact", "platform_native_artifact"}
            else "platform_additional"
        ),
        "normalization_timeout_budget_seconds": TIMEOUT_SECONDS["NORMALIZATION"],
        "normalization_memory_budget_mb": MEMORY_LIMIT_MB["NORMALIZATION"],
        "normalization_peak_rss_mb": payload.get("peak_rss_mb"),
        "normalization_status": payload.get("status"),
    }
    if payload.get("ok"):
        row.update({key: value for key, value in payload.items() if key not in {"ok", "traceback"}})
    else:
        row.update({
            "normalization_error_type": payload.get("error_type"),
            "normalization_error": payload.get("error"),
        })
        if payload.get("traceback"):
            row["normalization_traceback_file"] = save_trace(
                platform, case_id, "normalization", payload["traceback"]
            )
    NORMALIZED_ROWS.append(row)

normalized_df = pd.DataFrame(NORMALIZED_ROWS)
native_cols = [
    "run_id", "platform", "case_id", "status", "structural_status",
    "semantic_equivalence", "native_artifact_role", "logical_optimization_status",
    "canonical_semantic_status", "optimized_semantic_status",
    "recommended_semantic_status", "optimization_contract_status",
    "mapping_source", "mapping_sources",
    "native_qubits", "native_depth", "native_size", "native_ops",
    "artifact_bytes", "construction_seconds", "replay_seconds",
    "local_compile_seconds", "remote_materialization_seconds",
    "construction_peak_rss_mb", "replay_peak_rss_mb", "error_origin_stage",
]
native_df = results_df.reindex(columns=native_cols)

optimization_contract_df = results_df[results_df["platform"].eq("QDSV")].reindex(columns=[
    "case_id", "logical_optimization_status", "optimization_contract_status",
    "optimization_contract_errors", "optimizer_qiskit_version",
    "canonical_semantic_status", "optimized_semantic_status",
    "recommended_semantic_status", "native_artifact_role",
    "recommended_delivery_mode",
])

print("Vista primaria nativa — canonico QDSV y nativo Qrisp")
display(native_df)
print("Vistas QDSV separadas — canonica y optimizada, sin mezclarlas")
display(qdsv_views_df)
print("Auditoria del contrato de optimizacion logica QDSV")
display(optimization_contract_df)
print("Normalización externa protegida — evaluación posterior")
display(normalized_df)


# This table always exists. Resource comparison is enabled per case only
# when both canonical/native tracks are statistically consistent and normalized.
direct_rows = []
for case_id in PUBLIC_CASES:
    for platform in ("QDSV", "Qrisp"):
        result_row = results_df[
            results_df["platform"].eq(platform) & results_df["case_id"].eq(case_id)
        ]
        result = result_row.iloc[0].to_dict() if not result_row.empty else {}
        if platform == "QDSV":
            native_row = (
                qdsv_views_df[
                    qdsv_views_df["case_id"].eq(case_id)
                    & qdsv_views_df["artifact_role"].eq("canonical_ideal_artifact")
                ]
                if {"case_id", "artifact_role"}.issubset(qdsv_views_df.columns)
                else pd.DataFrame()
            )
            native = native_row.iloc[0].to_dict() if not native_row.empty else {}
            input_role = "canonical_ideal_artifact"
            validation = result.get("canonical_semantic_status")
        else:
            native = result
            input_role = "platform_native_artifact"
            validation = result.get("primary_semantic_status") or result.get("status")
        normalized_row = (
            normalized_df[
                normalized_df["platform"].eq(platform)
                & normalized_df["case_id"].eq(case_id)
                & normalized_df["input_artifact_role"].eq(input_role)
            ]
            if {"platform", "case_id", "input_artifact_role"}.issubset(normalized_df.columns)
            else pd.DataFrame()
        )
        normalized = normalized_row.iloc[0].to_dict() if not normalized_row.empty else {}
        direct_rows.append({
            "platform": platform,
            "case_id": case_id,
            "input_artifact_role": input_role,
            "validation": validation,
            "common_case_eligible": bool(result.get("common_case_eligible", False)),
            "verification_strength": (
                native.get("verification_strength")
                if platform == "QDSV"
                else result.get("verification_strength")
            ),
            "shots": native.get("shots") if platform == "QDSV" else result.get("shots"),
            "native_qubits": native.get("native_qubits"),
            "native_depth": native.get("native_depth"),
            "native_gates": native.get("native_size"),
            "normalized_status": normalized.get("normalization_status"),
            "normalized_qubits": normalized.get("normalized_qubits"),
            "normalized_depth": normalized.get("normalized_depth"),
            "normalized_cx": normalized.get("normalized_cx"),
            "construction_seconds": result.get("construction_seconds"),
            "operational_status": result.get("status"),
        })

common_case_direct_df = pd.DataFrame(direct_rows)
resource_comparison_eligible_by_case = {}
for case_id, frame in common_case_direct_df.groupby("case_id", sort=False):
    both_present = set(frame["platform"]) == {"QDSV", "Qrisp"}
    both_consistent = both_present and bool(frame["common_case_eligible"].all())
    both_normalized = both_present and bool(frame["normalized_status"].eq("normalized").all())
    resource_comparison_eligible_by_case[case_id] = both_consistent and both_normalized

common_case_direct_df["resource_comparison_eligible"] = common_case_direct_df["case_id"].map(
    resource_comparison_eligible_by_case
).fillna(False)
resource_comparison_eligible = bool(resource_comparison_eligible_by_case) and all(
    resource_comparison_eligible_by_case.values()
)
print("Comparación directa por microcaso")
display(common_case_direct_df)
print("Elegibilidad cuantitativa por caso:", resource_comparison_eligible_by_case)
print("Todos los casos habilitados:", resource_comparison_eligible)


## 16. Cobertura, paridad semántica y resultados incompletos


In [ ]:
coverage = (
    results_df
    .groupby(["platform", "status"], dropna=False)
    .size()
    .reset_index(name="cases")
)
coverage_pivot = (
    coverage
    .pivot(index="platform", columns="status", values="cases")
    .fillna(0)
    .astype(int)
)
display(coverage_pivot)

case_matrix = results_df.pivot(
    index="case_id",
    columns="platform",
    values="status",
)
display(case_matrix)

verification_matrix = results_df.pivot(
    index="case_id",
    columns="platform",
    values="verification_strength",
)
display(verification_matrix)

sampled_consistent_cases = results_df[
    results_df["primary_semantic_status"].eq("sampled_consistent")
][["platform", "case_id", "verification_strength"]]
print("Consistencia MPS muestreada:")
display(sampled_consistent_cases)

passed_counts = (
    results_df
    .assign(passed=results_df["common_case_eligible"].fillna(False).astype(bool))
    .groupby("case_id")["passed"]
    .sum()
)
comparable_cases = passed_counts[passed_counts >= 2].index.tolist()
bilateral_cases = passed_counts[passed_counts == 2].index.tolist()

ADAPTER_LIMIT_STATUSES = {"adapter_not_implemented", "adapter_error"}
OPERATIONAL_LIMIT_STATUSES = {
    "service_error",
    "quota_exhausted",
    "authorization_denied",
    "authentication_unavailable",
    "construction_timeout",
    "framework_installation_error",
    "construction_memory_limit",
    "replay_timeout",
    "replay_memory_limit",
    "normalization_timeout",
    "normalization_memory_limit",
    "worker_crash",
    "platform_preflight_blocked",
    "verification_incomplete",
    "optimization_contract_incomplete",
    "replay_not_available",
}
PLATFORM_OUTCOME_STATUSES = {
    "passed",
    "semantic_mismatch",
    "unsupported",
    "valid_rejection",
    "resource_rejected",
    "invalid_spec",
    "framework_construction_error",
    "artifact_unavailable",
    "artifact_export_error",
    "replay_mapping_required",
    "replay_resource_limited",
    "replay_error",
    "optimization_contract_failed",
    "sampled_consistent",
    "sampled_mismatch",
}

coverage_interpretation = []
for platform, frame in results_df.groupby("platform"):
    statuses = frame["status"]
    coverage_interpretation.append({
        "platform": platform,
        "total_cases": len(frame),
        "passed": int(frame["common_case_eligible"].fillna(False).astype(bool).sum()),
        "platform_outcomes_observed": int(statuses.isin(PLATFORM_OUTCOME_STATUSES).sum()),
        "adapter_gaps": int(statuses.isin(ADAPTER_LIMIT_STATUSES).sum()),
        "operational_incomplete": int(statuses.isin(OPERATIONAL_LIMIT_STATUSES).sum()),
        "semantic_failures": int(statuses.eq("semantic_mismatch").sum()),
    })
coverage_interpretation_df = pd.DataFrame(coverage_interpretation)
display(coverage_interpretation_df)

print("Casos comparables entre al menos dos plataformas:", comparable_cases)
print("Casos completados correctamente por ambas:", bilateral_cases)
print(
    "adapter_not_implemented y adapter_error no se contabilizan como incapacidad "
    "de la plataforma. Los bloqueos operativos tampoco se interpretan como unsupported."
)
if not bilateral_cases:
    print(
        "No se calculará un ranking global: no existe evidencia semántica "
        "completa de ambas plataformas para el microcaso."
    )


## 17. Indicadores de abstracción y experiencia del usuario final


In [ ]:
measured_effort = effort_df[effort_df["effort_status"].eq("measured")].copy()

effort_summary = (
    measured_effort
    .groupby("platform")
    .agg(
        shared_platform_infrastructure_loc=("shared_platform_infrastructure_loc", "first"),
        environment_compatibility_loc=("environment_compatibility_loc", "first"),
        measured_cases=("case_id", "count"),
        business_data_loc_median=("business_data_loc", "median"),
        neutral_predicate_specification_loc_median=("neutral_predicate_specification_loc", "median"),
        case_specific_platform_logic_loc_median=("case_specific_platform_logic_loc", "median"),
        end_user_invocation_loc_median=("end_user_invocation_loc", "median"),
        platform_specific_new_case_loc_median=("platform_specific_new_case_loc", "median"),
    )
    .reset_index()
)
display(pd.DataFrame([{
    "common_harness_loc": COMMON_HARNESS_LOC,
    "meaning": "Infraestructura neutral compartida; no atribuida a una plataforma",
}]))
display(effort_summary)

effort_by_case = measured_effort[[
    "platform", "case_id", "effort_status", "common_harness_loc",
    "business_data_loc", "neutral_predicate_specification_loc",
    "neutral_predicate_consumption", "shared_platform_infrastructure_loc",
    "case_specific_platform_logic_loc", "environment_compatibility_loc",
    "end_user_invocation_loc", "platform_specific_new_case_loc",
]].sort_values(["case_id", "platform"])
display(effort_by_case)

coverage_and_effort = coverage_interpretation_df.merge(effort_summary, on="platform", how="left")
display(coverage_and_effort)

print("Interpretación permitida:")
print("- Los datos y el significado del predicado son compartidos por necesidad experimental.")
print("- QDSV consume la especificación neutral mediante una capa reutilizable.")
print("- Qrisp reexpresa manualmente cada regla en lógica nativa específica.")
print("- La infraestructura reutilizable y la lógica por caso se reportan por separado.")
print("- No se calcula una métrica LOC única ni un ganador global de productividad.")


## 18. Exportar evidencia, manifiesto y paquete ZIP


In [ ]:
qrisp_phase_columns = [
    "platform", "case_id", "status", "construction_seconds",
    "qrisp_current_phase", "qrisp_current_native_event",
    "qrisp_elapsed_seconds_at_last_checkpoint",
    "qrisp_candidate_preparation_seconds", "qrisp_field_loading_seconds",
    "qrisp_condition_environment_seconds", "qrisp_field_cleanup_seconds",
    "qrisp_compile_seconds", "qrisp_qiskit_conversion_seconds",
    "qrisp_qpy_export_seconds", "qrisp_qasm_export_seconds",
    "qrisp_total_build_seconds",
]
qrisp_phase_timing_df = results_df.reindex(columns=qrisp_phase_columns)
qrisp_phase_timing_df = qrisp_phase_timing_df[
    qrisp_phase_timing_df["platform"].eq("Qrisp")
]
qrisp_phase_timing_df.to_csv(
    OUTPUT_DIR / "qrisp_phase_timing.csv", index=False
)

results_df.to_csv(OUTPUT_DIR / "platform_results.csv", index=False)
native_df.to_csv(OUTPUT_DIR / "native_metrics.csv", index=False)
normalized_df.to_csv(OUTPUT_DIR / "normalized_metrics.csv", index=False)
qdsv_views_df.to_csv(OUTPUT_DIR / "qdsv_native_artifact_views.csv", index=False)
optimization_contract_df.to_csv(OUTPUT_DIR / "qdsv_optimization_contract.csv", index=False)
effort_df.to_csv(OUTPUT_DIR / "user_effort_metrics.csv", index=False)
coverage_pivot.to_csv(OUTPUT_DIR / "coverage_matrix.csv")
case_matrix.to_csv(OUTPUT_DIR / "case_status_matrix.csv")
coverage_interpretation_df.to_csv(OUTPUT_DIR / "coverage_interpretation.csv", index=False)
common_case_direct_df.to_csv(OUTPUT_DIR / "common_case_direct_comparison.csv", index=False)
(OUTPUT_DIR / "common_case_direct_comparison.json").write_text(
    common_case_direct_df.to_json(orient="records", indent=2), encoding="utf-8"
)

(OUTPUT_DIR / "qdsv_native_artifact_views.json").write_text(
    json.dumps(QDSV_VIEW_ROWS, ensure_ascii=False, indent=2, default=repr), encoding="utf-8"
)
(OUTPUT_DIR / "platform_results.json").write_text(
    json.dumps(RUN_ROWS, ensure_ascii=False, indent=2, default=repr), encoding="utf-8"
)
(OUTPUT_DIR / "effort_records.json").write_text(
    json.dumps(EFFORT_ROWS, ensure_ascii=False, indent=2, default=repr), encoding="utf-8"
)
(OUTPUT_DIR / "resource_budgets.json").write_text(
    json.dumps(
        {
            "timeouts": TIMEOUT_SECONDS,
            "memory_mb": MEMORY_LIMIT_MB,
            "timeout_policy": TIMEOUT_POLICY,
        },
        indent=2,
    ), encoding="utf-8"
)

report = [
    "# Implementaciones independientes QDSV Bridge vs Qrisp nativo v19", "",
    f"- Run ID: `{RUN_ID}`",
    f"- Perfil principal: `{MAIN_ACCESS_PROFILE}`",
    f"- Politica de versiones: `{BENCHMARK_VERSION_POLICY}`",
    f"- Perfil onboarding: `{ONBOARDING_ACCESS_PROFILE}`",
    f"- Python: `{ENVIRONMENT_SNAPSHOT['python']}`", "",
    "## Alcance", "",
    "La comparación termina en el artefacto lógico ideal, QASM/Qiskit, evidencia y replay independiente.",
    "Construcción, replay y normalización tienen timeout y memoria separados.",
    "Esta corrida congela Bridge 0.6.1 y separa artefactos canonicos, optimizados nativos y normalizacion externa.",
    "El peak RSS es un guardrail absoluto del arbol de procesos e incluye paginas heredadas; no se usa para comparar plataformas.", "",
    "## Complejidad del caso", "",
    "Cada microcaso contiene exactamente una comparación pública; su complejidad se congela "
    "antes de construir los circuitos.", "",
    pd.DataFrame.from_dict(CASE_COMPLEXITY, orient="index").to_markdown(), "",
    "## Preflight operacional", "",
    "Los canarios solo controlan viabilidad operativa; no usan labels, respuestas esperadas ni resultados semanticos.", "",
    "``" + json.dumps(PLATFORM_PREFLIGHT_RECORDS, ensure_ascii=False) + "``", "",
    "## Tiempos por fase Qrisp", "",
    "Los checkpoints sobreviven a timeout o terminación del worker.", "",
    qrisp_phase_timing_df.to_markdown(index=False), "",
    "## Implementación Qrisp nativa", "",
    "Cada microcaso usa un ConditionEnvironment público explícito, sin AST QDSV ni expansión ANF.", "",
    "``" + json.dumps(QRISP_NATIVE_CASE_MANIFEST, ensure_ascii=False) + "``", "",
    "## Política de timeout", "",
    "El límite Qrisp de 1.200 s se aplica independientemente a cada microcaso y es un guardrail; "
    "no se interpreta como velocidad pura del framework.", "",
    "`construction_seconds` es latencia end-to-end observada, no velocidad pura de compilador.", "",
    "``" + json.dumps(TIMEOUT_POLICY, ensure_ascii=False) + "``", "",
    "## Controles de entorno", "",
    "Qrisp: ``" + json.dumps(QRISP_COMPATIBILITY_RECORD, ensure_ascii=False) + "``", "",
    "Estrategia Qrisp: `native_condition_environment`; preparación exacta del dominio; workspace 0; uncomputation habilitada.", "",
    "Contrato de independencia: ``" + json.dumps(PLATFORM_INDEPENDENCE_CONTRACT, ensure_ascii=False) + "``", "",
    "## Smoke test del README de Bridge", "",
    pd.DataFrame([README_SMOKE_RECORD]).to_markdown(index=False), "",
    "El smoke test se excluye de cobertura y recursos cruzados.", "",
    "## Vistas logicas nativas de QDSV", "", qdsv_views_df.to_markdown(index=False), "",
    "## Auditoria de optimizacion QDSV", "", optimization_contract_df.to_markdown(index=False), "",
    "## Cobertura por estado", "", coverage_pivot.to_markdown(), "",
    "## Interpretación de cobertura", "", coverage_interpretation_df.to_markdown(index=False), "",
    "## Matriz por caso", "", case_matrix.to_markdown(), "",
    "## Fuerza de validación semántica", "", verification_matrix.to_markdown(), "",
    "## Comparación directa del microcaso", "",
    common_case_direct_df.to_markdown(index=False), "",
    f"Elegibilidad cuantitativa por microcaso: {resource_comparison_eligible_by_case}", "",
    f"Todos los microcasos habilitados: {resource_comparison_eligible}", "",
    "## Casos comparables", "",
    f"Al menos dos plataformas: {comparable_cases}", "",
    f"Ambas plataformas: {bilateral_cases}", "",
    "## Esfuerzo verificable", "",
    f"Harness común no atribuible: {COMMON_HARNESS_LOC} LOC", "",
    "Política LOC: ``" + json.dumps(LOC_ATTRIBUTION_POLICY, ensure_ascii=False) + "``", "",
    "Resumen por plataforma:", "", effort_summary.to_markdown(index=False), "",
    "Atribución por caso:", "", effort_by_case.to_markdown(index=False), "",
    "## Regla de interpretación", "",
    "No se declara ganador global.",
    "Los recursos solo se comparan si ambas plataformas son sampled_consistent y la normalización común termina.",
    "Las brechas del adaptador, autenticación, soporte, recursos, memoria y corrección permanecen separadas.",
]
(OUTPUT_DIR / "report.md").write_text("\n".join(report), encoding="utf-8")
display(Markdown((OUTPUT_DIR / "report.md").read_text(encoding="utf-8")))


def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

manifest_entries = []
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file() and path.name != "manifest.json":
        manifest_entries.append({
            "relative_path": str(path.relative_to(OUTPUT_DIR)),
            "bytes": path.stat().st_size,
            "sha256": file_sha256(path),
        })
manifest = {
    "run_id": RUN_ID,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "output_directory": str(OUTPUT_DIR),
    "file_count_excluding_manifest": len(manifest_entries),
    "files": manifest_entries,
}
(OUTPUT_DIR / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

archive_base = OUTPUT_ROOT / f"benchmark_qdsv_qrisp_v19_independent_native_{RUN_ID}"
archive = shutil.make_archive(
    str(archive_base), "zip", root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name
)
print("Paquete generado:", archive)
print("El ZIP contiene únicamente el directorio de esta corrida:", RUN_ID)
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", path.name)

try:
    from google.colab import files
    files.download(archive)
except Exception:
    print("Descarga automática no disponible; archivo:", archive)


## Interpretación permitida

Esta versión compara implementaciones independientes específicas de cada plataforma mediante sus interfaces públicas. No afirma que operen sin adaptadores o sin código local.

La generación QDSV se delega a `qdsv-bridge`; el notebook contiene una capa reutilizable que forma la especificación pública. Qrisp utiliza su API nativa con infraestructura reversible compartida y una condición manual específica para cada regla empresarial.

Ambas plataformas comparten los datos y el significado semántico público. El ground truth y el harness común aparecen solo después de la construcción y no forman parte de los circuitos.

Las métricas LOC deben leerse por categoría. En particular, `case_specific_platform_logic_loc` impide atribuir las ramas Qrisp específicas al adaptador reutilizable. No se calcula un total único de productividad ni un ganador global.

El track debe denominarse **Qrisp 0.9.6 native public API + audited packaging compatibility shim**. El resultado caracteriza esta implementación pública concreta, no toda la capacidad teórica de Qrisp.

La comparación de recursos usa exclusivamente el artefacto canónico de QDSV y el nativo de Qrisp, y solo se habilita cuando ambos son consistentes y están normalizados. `construction_seconds` es latencia end-to-end observada, no velocidad pura de compilador.
